# 3D reporter timelapse — 05d_reporter_alignment_switchlike

**Feeds:** Fig 5h

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# Reporter Alignment And Switch-Like Views

This notebook extracts Theme 3 from the broader cooperativity exploration.

The goal here is to test whether the raw clock-time `YFP` mean is hiding a sharper shared transition because different positions switch at different times.

Notebook `05b` remains Theme 1, and notebook `05c` carries the direct temporal-ordering analyses.


## Interpretation Frame

The main questions here are:

1. does aligning positions to their own `YFP` transition sharpen the aggregate `YFP` progression?
2. after alignment, does `YFP` look more switch-like than the raw mean suggests?
3. are the aligned derivative and rate-versus-state views cleaner and more interpretable than the clock-time versions?

To keep reruns lighter, this notebook reuses shared half-max and aligned-trace caches in `results/tables`.


## Upstream Signal Preprocessing

The thresholded reporter metrics shown here are inherited from notebook `05`; this notebook does **not** reprocess images from scratch.

The upstream preprocessing chain is:

1. apply reporter-specific illumination correction to the raw `FOXF1-RFP` and `BMP4-YFP` images using the shared channel-wide illumination fields estimated earlier in the pipeline
2. for each image/frame separately, measure the whole off-cyst pixel median **after** illumination correction and subtract that per-image scalar background
3. after that subtraction, estimate the reporter-negative within-cyst baseline and sigma from early corrected cyst pixels, then call positive pixels at the chosen `N sigma` threshold

So the thresholded areas, positive fractions, and positive-region intensity summaries in these notebooks all sit on top of:

- illumination correction
- per-image off-cyst median background subtraction
- within-cyst baseline / sigma estimation for thresholding

By default in the manuscript-facing reporter notebooks, the thresholded positive-fraction comparisons use:

- `FOXF1-RFP`: `4 sigma`
- `BMP4-YFP`: `3 sigma`


## Setup


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from matplotlib.ticker import FuncFormatter

pd.set_option("display.max_columns", 200)
plt.rcParams["figure.dpi"] = 120


In [ ]:
cwd = Path.cwd().resolve()
root_candidates = [cwd] + list(cwd.parents[:3])
ROOT = None
for candidate in root_candidates:
    if (candidate / "data").exists() and (candidate / "results").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Could not locate project root from current working directory.")

qc_dir = ROOT / "scripts" / "qc"
if str(qc_dir) not in sys.path:
    sys.path.insert(0, str(qc_dir))

from notebook_display_helpers import (
    TIME_DISPLAY_OFFSET_HOURS,
    display_time_hours as _display_time_hours,
    format_display_hours as _format_display_hours,
    offset_time_hours_df as _offset_time_hours_df,
    set_display_time_axis as _set_display_time_axis,
)
from notebook_invariant_helpers import (
    THEME_DEFAULT_SIGMA_BY_REPORTER,
    assert_metric_name_sigma_consistency,
    assert_no_excluded_keys_in_analysis,
    assert_theme_default_sigmas,
)
from reporter_cooperativity_shared import (
    COMMON_HALFMAX_METRICS,
    FIT_COLORS,
    METRIC_LABELS,
    REPORTER_COLORS,
    aggregate_aligned_mean,
    aggregate_mean_trace,
    build_halfmax_lookup,
    fit_linear_and_logistic,
    focus_ylim_from_arrays,
    gaussian_smooth,
    halfmax_time_for_position,
    halfmax_time_from_lookup,
    load_basic_inputs,
    load_or_build_aligned_pair_cache,
    load_or_build_halfmax_cache,
    local_gradient,
    normalize_trace,
)

ROOT, population_metrics, global_thresholds = load_basic_inputs(ROOT)
FIGURE_DIR = ROOT / "results" / "figures" / "05d"
TABLE_DIR = ROOT / "results" / "tables"
FRAME_METRICS_PATH = TABLE_DIR / "05_reporter_metrics_by_frame.tsv"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DEFAULT_RFP_SIGMA = int(THEME_DEFAULT_SIGMA_BY_REPORTER["RFP"])
DEFAULT_YFP_SIGMA = int(THEME_DEFAULT_SIGMA_BY_REPORTER["YFP"])
DEFAULT_POSITIVE_FRACTION_METRICS = {
    "RFP": f"positive_fraction_sigma{DEFAULT_RFP_SIGMA}",
    "YFP": f"positive_fraction_sigma{DEFAULT_YFP_SIGMA}",
}
DEFAULT_POSITIVE_MEAN_METRICS = {
    "RFP": f"positive_mean_intensity_sigma{DEFAULT_RFP_SIGMA}_z",
    "YFP": f"positive_mean_intensity_sigma{DEFAULT_YFP_SIGMA}_z",
}
frame_metrics = pd.read_csv(
    FRAME_METRICS_PATH,
    sep="\t",
    low_memory=False,
    usecols=["position_label", "time_index", "reporter", "exclude_from_analysis"],
)

assert_theme_default_sigmas(
    {"RFP": DEFAULT_RFP_SIGMA, "YFP": DEFAULT_YFP_SIGMA},
    context="05d Theme 3 default thresholds",
)
assert_metric_name_sigma_consistency(
    DEFAULT_POSITIVE_FRACTION_METRICS,
    context="05d default positive-fraction pairing",
)
assert_metric_name_sigma_consistency(
    DEFAULT_POSITIVE_MEAN_METRICS,
    context="05d default positive-mean pairing",
)
assert_no_excluded_keys_in_analysis(
    frame_metrics,
    population_metrics,
    key_columns=("position_label", "time_index", "reporter"),
    context="05d population metrics",
)

HALFMAX_CACHE_PATH = TABLE_DIR / "05_cooperativity_halfmax_times.tsv"
ALIGNED_FRACTION_CACHE_PATH = TABLE_DIR / "05_cooperativity_aligned_positive_fraction_rfp4_yfp3_yfp_halfmax.tsv"
ALIGNED_MEAN_CACHE_PATH = TABLE_DIR / "05_cooperativity_aligned_positive_mean_rfp4_yfp3_z_yfp_halfmax.tsv"

halfmax_cache_existed = HALFMAX_CACHE_PATH.exists()
aligned_fraction_cache_existed = ALIGNED_FRACTION_CACHE_PATH.exists()
aligned_mean_cache_existed = ALIGNED_MEAN_CACHE_PATH.exists()

halfmax_df = load_or_build_halfmax_cache(
    population_metrics=population_metrics,
    cache_path=HALFMAX_CACHE_PATH,
    metric_names=COMMON_HALFMAX_METRICS,
    smooth_sigma=4.0,
)
halfmax_lookup = build_halfmax_lookup(halfmax_df)

aligned_fraction_df = load_or_build_aligned_pair_cache(
    population_metrics=population_metrics,
    cache_path=ALIGNED_FRACTION_CACHE_PATH,
    metric_name_by_reporter=DEFAULT_POSITIVE_FRACTION_METRICS,
    anchor_metric_name="positive_fraction_sigma3",
    anchor_reporter="YFP",
    halfmax_lookup=halfmax_lookup,
)
aligned_mean_df = load_or_build_aligned_pair_cache(
    population_metrics=population_metrics,
    cache_path=ALIGNED_MEAN_CACHE_PATH,
    metric_name_by_reporter=DEFAULT_POSITIVE_MEAN_METRICS,
    anchor_metric_name="positive_mean_intensity_sigma3_z",
    anchor_reporter="YFP",
    halfmax_lookup=halfmax_lookup,
)

display(
    Markdown(
        f'''
        **Loaded shared cooperativity caches**

        - population trace rows: `{len(population_metrics):,}`
        - shared half-max cache: `{"reused existing" if halfmax_cache_existed else "created new"}`
        - aligned positive-fraction cache: `{"reused existing" if aligned_fraction_cache_existed else "created new"}`
        - aligned positive-mean cache: `{"reused existing" if aligned_mean_cache_existed else "created new"}`
        - default thresholded pairing for Theme 3:
          - `RFP`: `{DEFAULT_RFP_SIGMA} sigma`
          - `YFP`: `{DEFAULT_YFP_SIGMA} sigma`
        '''
    )
)


In [ ]:
def should_offset_time_column(column_name: str) -> bool:
    column = str(column_name)
    if not column.endswith("_hours"):
        return False
    blocked_tokens = ["relative_", "lag_", "rise_", "interval_"]
    return not any(token in column for token in blocked_tokens)


def display_time_hours(values):
    return _display_time_hours(values, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def format_display_hours(value: float, decimals: int = 1) -> str:
    return _format_display_hours(value, decimals=decimals, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def set_display_time_axis(ax, axis: str = "x", crowded: bool = False) -> None:
    _set_display_time_axis(ax, axis=axis, crowded=crowded, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def display_time_df(df: pd.DataFrame) -> pd.DataFrame:
    return _offset_time_hours_df(
        df,
        should_offset_column=should_offset_time_column,
        offset_hours=TIME_DISPLAY_OFFSET_HOURS,
    )


def render_aligned_mean(aligned_df: pd.DataFrame, figure_path: Path, figure_title: str) -> None:
    fig, ax = plt.subplots(figsize=(7.5, 4.3), constrained_layout=True)
    plotted = []
    for reporter in ["RFP", "YFP"]:
        summary = aggregate_aligned_mean(aligned_df, reporter)
        plotted.append(summary["normalized_value"].to_numpy(dtype=float))
        ax.plot(
            summary["relative_hours"],
            summary["normalized_value"],
            color=REPORTER_COLORS[reporter],
            linewidth=2.7,
            label=reporter,
        )
    ax.axvline(0.0, color="0.55", linestyle="--", linewidth=1.2)
    ax.set_xlabel("Hours relative to YFP half-max")
    ax.set_ylabel("Normalized reporter progression")
    ax.set_ylim(focus_ylim_from_arrays(plotted, include_zero=True, padding_fraction=0.08))
    ax.grid(alpha=0.18)
    ax.legend(loc="upper left", frameon=False)
    ax.set_title(figure_title, fontsize=11.0)
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote figure:", figure_path)


def render_individual_yfp_trace_grid(
    metric_name: str,
    metric_title: str,
    ylabel: str,
    figure_path: Path,
    include_zero: bool = True,
    halfmax_smooth_sigma: float = 4.0,
    exclude_positions: tuple[str, ...] = (),
    normalize_by_amplitude: bool = False,
    normalization_mode: str = "baseline_peak",
    normalization_smooth_sigma: float = 4.0,
    normalization_early_n: int = 8,
    y_limits_override: tuple[float, float] | None = None,
    robust_ylim_quantiles: tuple[float, float] | None = (0.01, 0.99),
    n_cols: int = 5,
    panel_width: float = 2.5,
    panel_height: float = 1.8,
    use_panel_legends: bool = False,
    show_halfmax_in_panel_label: bool = True,
    show_scale_in_panel_label: bool = False,
    show_figure_title: bool = True,
    max_scale_factor: float | None = None,
    sort_by: str = "halfmax_time",
    sort_descending: bool = False,
    line_width: float = 1.8,
    label_fontsize: float = 7.5,
    tick_fontsize: float = 6.0,
) -> None:
    excluded = {str(position_label) for position_label in exclude_positions}
    filtered_population_metrics = population_metrics.loc[
        (population_metrics["reporter"] == "YFP")
        & (~population_metrics["position_label"].isin(excluded))
    ].copy()
    trace_rows = []
    for position_label, subset in (
        filtered_population_metrics.loc[:, ["position_label", "time_hours", metric_name]]
        .groupby("position_label", sort=True)
    ):
        ordered = subset.sort_values("time_hours")
        raw_values = ordered[metric_name].to_numpy(dtype=float)
        plotted_values = raw_values.copy()
        amplitude = float("nan")
        if normalize_by_amplitude:
            if normalization_mode == "minmax":
                finite_raw = raw_values[np.isfinite(raw_values)]
                if finite_raw.size >= 2:
                    baseline = float(np.nanmin(finite_raw))
                    peak = float(np.nanmax(finite_raw))
                    amplitude = peak - baseline
                else:
                    amplitude = float("nan")
            else:
                smoothed = gaussian_smooth(raw_values, normalization_smooth_sigma)
                finite_smoothed = smoothed[np.isfinite(smoothed)]
                if finite_smoothed.size >= max(normalization_early_n, 4):
                    baseline = float(np.nanmedian(smoothed[:normalization_early_n]))
                    peak = float(np.nanmax(smoothed))
                    amplitude = peak - baseline
                else:
                    amplitude = float("nan")
            if np.isfinite(amplitude) and amplitude > 0:
                plotted_values = (raw_values - baseline) / amplitude
            else:
                plotted_values = np.full_like(raw_values, np.nan, dtype=float)
        trace_rows.append(
            {
                "position_label": position_label,
                "halfmax_time_hours": halfmax_time_for_position(
                    filtered_population_metrics,
                    metric_name,
                    position_label,
                    "YFP",
                    smooth_sigma=halfmax_smooth_sigma,
                ),
                "time_hours": ordered["time_hours"].to_numpy(dtype=float),
                "values": plotted_values,
                "scale_factor": (1.0 / amplitude) if np.isfinite(amplitude) and amplitude > 0 else float("nan"),
            }
        )

    if max_scale_factor is not None:
        trace_rows = [
            row
            for row in trace_rows
            if (not np.isfinite(row["scale_factor"])) or (row["scale_factor"] <= max_scale_factor)
        ]

    trace_rows = [
        row
        for row in trace_rows
        if np.isfinite(row["values"]).any()
    ]

    if sort_by == "scale_factor":
        trace_rows = sorted(
            trace_rows,
            key=lambda row: (
                not np.isfinite(row["scale_factor"]),
                -row["scale_factor"] if sort_descending and np.isfinite(row["scale_factor"]) else row["scale_factor"],
                row["position_label"],
            ),
        )
    else:
        trace_rows = sorted(
            trace_rows,
            key=lambda row: (
                not np.isfinite(row["halfmax_time_hours"]),
                -row["halfmax_time_hours"] if sort_descending and np.isfinite(row["halfmax_time_hours"]) else row["halfmax_time_hours"],
                row["position_label"],
            ),
        )

    n_panels = len(trace_rows)
    n_rows = int(np.ceil(n_panels / n_cols))
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(panel_width * n_cols, panel_height * n_rows),
        sharex=True,
        sharey=True,
    )
    axes = np.asarray(axes).reshape(-1)

    all_values = [row["values"] for row in trace_rows if np.isfinite(row["values"]).any()]
    if not all_values:
        all_values = [np.asarray([0.0, 1.0], dtype=float)]
    if y_limits_override is not None:
        y_limits = y_limits_override
    elif robust_ylim_quantiles is not None:
        pooled = np.concatenate([arr[np.isfinite(arr)] for arr in all_values if np.isfinite(arr).any()])
        if pooled.size:
            lower = float(np.nanquantile(pooled, robust_ylim_quantiles[0]))
            upper = float(np.nanquantile(pooled, robust_ylim_quantiles[1]))
            if include_zero:
                lower = min(lower, 0.0)
                upper = max(upper, 0.0)
            span = upper - lower
            if not np.isfinite(span) or span <= 0:
                span = max(abs(upper - lower), 1.0)
            pad = max(1e-6, 0.06 * span)
            y_limits = (lower - pad, upper + pad)
        else:
            y_limits = focus_ylim_from_arrays(all_values, include_zero=include_zero, padding_fraction=0.06)
    else:
        y_limits = focus_ylim_from_arrays(all_values, include_zero=include_zero, padding_fraction=0.06)

    for ax, row in zip(axes, trace_rows):
        finite = np.isfinite(row["time_hours"]) & np.isfinite(row["values"])
        line = None
        if finite.sum() >= 2:
            line = ax.plot(
                row["time_hours"][finite],
                row["values"][finite],
                color=REPORTER_COLORS["YFP"],
                linewidth=line_width,
            )[0]
        label_parts = [row["position_label"]]
        if show_halfmax_in_panel_label:
            halfmax_label = (
                f"t1/2={format_display_hours(row['halfmax_time_hours'], 1)}"
                if np.isfinite(row["halfmax_time_hours"])
                else "t1/2=NA"
            )
            label_parts.append(halfmax_label)
        if show_scale_in_panel_label:
            scale_label = (
                f"scale x{row['scale_factor']:.1f}"
                if np.isfinite(row["scale_factor"])
                else "scale xNA"
            )
            label_parts.append(scale_label)
        panel_label = " | ".join(label_parts)
        if use_panel_legends and line is not None:
            ax.legend(
                [line],
                [panel_label],
                loc="upper left",
                frameon=False,
                fontsize=label_fontsize,
                handlelength=0.9,
                handletextpad=0.3,
                borderaxespad=0.15,
            )
        else:
            ax.set_title(panel_label, fontsize=label_fontsize)
        ax.set_ylim(y_limits)
        ax.grid(alpha=0.16)
        ax.tick_params(labelsize=tick_fontsize, length=2.0, pad=1.5)

    for ax in axes[n_panels:]:
        ax.axis("off")

    for ax in axes:
        ax.label_outer()
        set_display_time_axis(ax, "x")

    fig.supxlabel("Time (hours)", fontsize=8.5)
    fig.supylabel(ylabel, fontsize=8.5)
    if show_figure_title:
        fig.suptitle(metric_title, fontsize=11.6)
        top_margin = 0.975
    else:
        top_margin = 0.995
    fig.subplots_adjust(left=0.08, right=0.995, bottom=0.06, top=top_margin, wspace=0.05, hspace=0.08)
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote figure:", figure_path)


## Theme 3: After Alignment, YFP Looks More Switch-Like Than The Raw Mean Suggests


### Main Figure Candidate

These are the 1-2 plots that most directly answer Theme 3.

Plot 1 asks whether the aligned `YFP` summary is more switch-like than the raw clock-time mean.

Plot 2 shows the position-level heterogeneity that can flatten the raw mean and the sharper shared transition that appears after alignment.


#### Plot 1. Absolute-Versus-Aligned YFP Sigmoid View

This is the clearest single summary for the theme.

It is still phenomenological, but it makes the “raw mean hides a sharper transition” idea very explicit.


In [ ]:
sigmoid_metric_specs = [
    {
        "yfp_metric_name": "positive_fraction_sigma3",
        "aligned_df": aligned_fraction_df,
        "metric_title": "Positive fraction (YFP 3 sigma mask)",
    },
    {
        "yfp_metric_name": "positive_mean_intensity_sigma3_z",
        "aligned_df": aligned_mean_df,
        "metric_title": "Positive-region mean intensity (YFP 3 sigma mask)",
    },
]

sigmoid_summary_rows = []
fit_results = {}
for spec in sigmoid_metric_specs:
    absolute_summary = aggregate_mean_trace(population_metrics, spec["yfp_metric_name"], "YFP")
    absolute_x = absolute_summary["time_hours"].to_numpy(dtype=float)
    absolute_y = normalize_trace(absolute_summary["mean"].to_numpy(dtype=float), early_n=8, smooth_sigma=4.0)
    absolute_fit = fit_linear_and_logistic(absolute_x, absolute_y)

    aligned_summary = aggregate_aligned_mean(spec["aligned_df"], "YFP")
    aligned_x = aligned_summary["relative_hours"].to_numpy(dtype=float)
    aligned_y = aligned_summary["normalized_value"].to_numpy(dtype=float)
    aligned_fit = fit_linear_and_logistic(aligned_x, aligned_y)

    fit_results[spec["yfp_metric_name"]] = {
        "metric_title": spec["metric_title"],
        "absolute_fit": absolute_fit,
        "aligned_fit": aligned_fit,
    }

    for context_label, fit_info in [("absolute", absolute_fit), ("aligned", aligned_fit)]:
        improvement = float("nan")
        if np.isfinite(fit_info["linear_rss"]) and np.isfinite(fit_info["logistic_rss"]) and fit_info["logistic_rss"] > 0:
            improvement = float(fit_info["linear_rss"] / fit_info["logistic_rss"])
        sigmoid_summary_rows.append(
            {
                "yfp_metric_name": spec["yfp_metric_name"],
                "metric_label": spec["metric_title"],
                "context": context_label,
                "linear_rss": float(fit_info["linear_rss"]),
                "logistic_rss": float(fit_info["logistic_rss"]),
                "rss_improvement_factor": improvement,
                "logistic_midpoint_hours": float(fit_info["logistic_midpoint"]),
                "logistic_slope_per_hour": float(fit_info["logistic_slope"]),
                "logistic_rise_10_90_hours": float(fit_info["logistic_rise_10_90_hours"]),
            }
        )

sigmoid_summary_df = pd.DataFrame(sigmoid_summary_rows)
sigmoid_summary_path = TABLE_DIR / "05d_yfp_sigmoid_fit_summary.tsv"
display_time_df(sigmoid_summary_df).to_csv(sigmoid_summary_path, sep="\t", index=False)

fig, axes = plt.subplots(2, 2, figsize=(10.2, 7.6), constrained_layout=True)
for row_idx, spec in enumerate(sigmoid_metric_specs):
    absolute_fit = fit_results[spec["yfp_metric_name"]]["absolute_fit"]
    aligned_fit = fit_results[spec["yfp_metric_name"]]["aligned_fit"]
    for context_label, fit_info, ax in [
        ("absolute", absolute_fit, axes[row_idx, 0]),
        ("aligned", aligned_fit, axes[row_idx, 1]),
    ]:
        ax.plot(fit_info["x"], fit_info["y"], color=FIT_COLORS["mean"], linewidth=2.7, label="mean progression")
        ax.plot(fit_info["x"], fit_info["linear_pred"], color=FIT_COLORS["linear"], linewidth=2.0, label="linear fit")
        ax.plot(fit_info["x"], fit_info["logistic_pred"], color=FIT_COLORS["logistic"], linewidth=2.2, label="logistic fit")
        if context_label == "aligned":
            ax.axvline(0.0, color="0.55", linestyle="--", linewidth=1.0)
            ax.set_xlabel("Hours relative to YFP half-max")
        else:
            ax.set_xlabel("Time (hours)")
            set_display_time_axis(ax, "x")
        ax.set_ylim(-0.05, 1.08)
        ax.set_ylabel("Normalized YFP progression")
        ax.set_title(f"{spec['metric_title']} | {context_label}", fontsize=9.7)
        ax.grid(alpha=0.18)
        if row_idx == 0 and context_label == "absolute":
            ax.legend(loc="upper left", frameon=False)

        improvement = float("nan")
        if np.isfinite(fit_info["linear_rss"]) and np.isfinite(fit_info["logistic_rss"]) and fit_info["logistic_rss"] > 0:
            improvement = float(fit_info["linear_rss"] / fit_info["logistic_rss"])
        ax.text(
            0.97,
            0.03,
            "\n".join(
                [
                    f"linear RSS = {fit_info['linear_rss']:.3f}",
                    f"logistic RSS = {fit_info['logistic_rss']:.3f}",
                    f"improvement = {improvement:.2f}x",
                    f"logistic 10-90 rise = {fit_info['logistic_rise_10_90_hours']:.1f} h",
                ]
            ),
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=7.9,
            bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "0.8", "alpha": 0.92},
        )

sigmoid_figure_path = FIGURE_DIR / "05d_yfp_sigmoid_fit_absolute_vs_aligned.png"
fig.suptitle("YFP progression in clock time versus after alignment", fontsize=11.6)
fig.savefig(sigmoid_figure_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", sigmoid_figure_path)
print("Wrote table:", sigmoid_summary_path)


#### Plot 2. Positive-Fraction Aligned Heatmaps

This plot shows the position-level aligned progression directly.

Sorting positions by `RFP` progression at aligned time 0 helps show how much `RFP` context is already in place when `YFP` reaches its own transition point.


In [ ]:
heatmap_relative_hours = np.arange(-12.0, 12.0 + 0.25, 0.25, dtype=float)
heatmap_rows = []
for position_label, _ in population_metrics.groupby("position_label", sort=True):
    yfp_anchor = halfmax_time_from_lookup(halfmax_lookup, "positive_fraction_sigma3", position_label, "YFP")
    if not np.isfinite(yfp_anchor):
        continue
    entry = {"position_label": position_label, "rfp_state_at_anchor": np.nan}
    for reporter in ["RFP", "YFP"]:
        subset = population_metrics.loc[
            (population_metrics["position_label"] == position_label)
            & (population_metrics["reporter"] == reporter)
        ].sort_values("time_hours")
        time_hours = subset["time_hours"].to_numpy(dtype=float)
        metric_name = "positive_fraction_sigma4" if reporter == "RFP" else "positive_fraction_sigma3"
        normalized_values = normalize_trace(subset[metric_name].to_numpy(dtype=float), early_n=8, smooth_sigma=4.0)
        finite = np.isfinite(time_hours) & np.isfinite(normalized_values)
        if finite.sum() < 12:
            continue
        valid_time = time_hours[finite] - yfp_anchor
        valid_values = normalized_values[finite]
        interpolated = np.full_like(heatmap_relative_hours, np.nan, dtype=float)
        inside = (heatmap_relative_hours >= valid_time.min()) & (heatmap_relative_hours <= valid_time.max())
        interpolated[inside] = np.interp(heatmap_relative_hours[inside], valid_time, valid_values)
        entry[reporter] = interpolated
        if reporter == "RFP" and 0.0 >= valid_time.min() and 0.0 <= valid_time.max():
            entry["rfp_state_at_anchor"] = float(np.interp(0.0, valid_time, valid_values))
    if np.isfinite(entry["rfp_state_at_anchor"]) and "RFP" in entry and "YFP" in entry:
        heatmap_rows.append(entry)

heatmap_rows = sorted(heatmap_rows, key=lambda row: row["rfp_state_at_anchor"])

heatmap_fig, heatmap_axes = plt.subplots(1, 2, figsize=(10.0, 7.3), sharex=True, sharey=True, constrained_layout=True)
heatmap_cmap = plt.cm.magma.copy()
heatmap_cmap.set_bad(color="#d0d0d0")
for ax, reporter in zip(heatmap_axes, ["RFP", "YFP"]):
    matrix = np.vstack([row[reporter] for row in heatmap_rows])
    image = ax.imshow(
        matrix,
        aspect="auto",
        origin="lower",
        extent=[heatmap_relative_hours.min(), heatmap_relative_hours.max(), 0, matrix.shape[0]],
        vmin=0.0,
        vmax=1.0,
        cmap=heatmap_cmap,
    )
    ax.axvline(0.0, color="cyan", linestyle="--", linewidth=1.0)
    ax.set_title(reporter, fontsize=10.2)
    ax.set_xlabel("Hours relative to YFP half-max")
heatmap_axes[0].set_ylabel("Positions (sorted by RFP progression at t = 0)")
heatmap_fig.colorbar(image, ax=heatmap_axes.ravel().tolist(), shrink=0.82, label="Normalized progression")
aligned_heatmap_path = FIGURE_DIR / "05d_positive_fraction_aligned_heatmaps.png"
heatmap_fig.suptitle("Positive-fraction aligned heatmaps", fontsize=11.6)
heatmap_fig.savefig(aligned_heatmap_path, dpi=180, bbox_inches="tight")
display(heatmap_fig)
plt.close(heatmap_fig)
print("Wrote figure:", aligned_heatmap_path)


### Supplementary Figures


#### Raw Aggregate YFP Views In Clock Time

This is the most literal starting point for Theme 3.

Before any alignment, we should first look directly at the aggregate `YFP` activity summaries in clock time and judge what the unaligned population mean actually looks like.


##### Plot 3. Aggregate YFP Activity In Clock Time

These raw clock-time means are shown without alignment so the starting point is visible before any transition-centering or fitted comparison.

The second row adds within-cyst normalized versions of all four summaries. For positive fraction, each trace is min-max normalized to `[0, 1]` using that cyst's raw minimum and maximum. For the three intensity summaries, baseline and peak are estimated from a Gaussian-smoothed copy of that same trace (`sigma = 4` frames; baseline = median of the first 8 smoothed timepoints; peak = maximum of the smoothed trace), then the displayed raw trace is rescaled as `(raw - baseline) / (peak - baseline)` with no clipping. Gray traces requiring more than `10x` rescaling are omitted for readability. The gray traces are normalized cyst-by-cyst, while the gold line is the same aggregate mean trace as in the top row, normalized only once as a single trace so its clock-time shape is preserved.


In [ ]:
raw_yfp_specs = [
    {
        "metric_name": "positive_fraction_sigma3",
        "metric_title": "Positive fraction (YFP 3 sigma mask)",
        "ylabel": "Mean positive fraction",
        "include_zero": True,
        "padding_fraction": 0.10,
        "normalized_ylabel": "Normalized positive fraction",
        "normalization_mode": "minmax",
        "normalized_ylim": (0.0, 1.0),
        "normalized_max_scale_factor": 10.0,
    },
    {
        "metric_name": "positive_mean_intensity_sigma3_z",
        "metric_title": "Positive-region mean intensity (YFP 3 sigma mask, Pos42 omitted)",
        "ylabel": "Mean sigma above pooled null",
        "include_zero": False,
        "padding_fraction": 0.05,
        "exclude_positions": ("Pos42",),
        "normalized_ylabel": "Normalized intensity",
        "normalization_mode": "baseline_peak",
        "normalized_ylim": (-0.5, 1.5),
        "normalized_max_scale_factor": 10.0,
    },
    {
        "metric_name": "organoid_mean_intensity_z",
        "metric_title": "Whole-cyst mean intensity",
        "ylabel": "Mean sigma above pooled null",
        "include_zero": False,
        "padding_fraction": 0.05,
        "normalized_ylabel": "Normalized intensity",
        "normalization_mode": "baseline_peak",
        "normalized_ylim": (-0.5, 1.5),
        "normalized_max_scale_factor": 10.0,
    },
    {
        "metric_name": "brightest_decile_mean_intensity_z",
        "metric_title": "Brightest 10% mean intensity",
        "ylabel": "Mean sigma above pooled null",
        "include_zero": False,
        "padding_fraction": 0.05,
        "normalized_ylabel": "Normalized intensity",
        "normalization_mode": "baseline_peak",
        "normalized_ylim": (-0.5, 1.5),
        "normalized_max_scale_factor": 10.0,
    },
]

def normalize_raw_trace_for_display(
    values: np.ndarray,
    mode: str = "baseline_peak",
    smooth_sigma: float = 4.0,
    early_n: int = 8,
) -> tuple[np.ndarray, float]:
    arr = np.asarray(values, dtype=float)
    if mode == "minmax":
        finite_arr = arr[np.isfinite(arr)]
        if finite_arr.size < 2:
            return np.full_like(arr, np.nan, dtype=float), float("nan")
        baseline = float(np.nanmin(finite_arr))
        peak = float(np.nanmax(finite_arr))
        amplitude = peak - baseline
        if not np.isfinite(amplitude) or amplitude <= 0:
            return np.full_like(arr, np.nan, dtype=float), float("nan")
        return (arr - baseline) / amplitude, float(1.0 / amplitude)
    smoothed = gaussian_smooth(arr, smooth_sigma)
    finite_smoothed = smoothed[np.isfinite(smoothed)]
    if finite_smoothed.size < max(early_n, 4):
        return np.full_like(arr, np.nan, dtype=float), float("nan")
    baseline = float(np.nanmedian(smoothed[:early_n]))
    peak = float(np.nanmax(smoothed))
    amplitude = peak - baseline
    if not np.isfinite(amplitude) or amplitude <= 0:
        return np.full_like(arr, np.nan, dtype=float), float("nan")
    return (arr - baseline) / amplitude, float(1.0 / amplitude)

raw_yfp_clock_time_path = FIGURE_DIR / "05d_yfp_activity_clock_time.png"
fig, axes = plt.subplots(
    2,
    len(raw_yfp_specs),
    figsize=(5.2 * len(raw_yfp_specs), 8.0),
    sharex="col",
    constrained_layout=True,
)
for ax, spec in zip(axes[0], raw_yfp_specs):
    excluded = {str(position_label) for position_label in spec.get("exclude_positions", ())}
    plot_df = population_metrics.loc[
        (population_metrics["reporter"] == "YFP")
        & (~population_metrics["position_label"].isin(excluded))
    ].copy()
    per_position = (
        plot_df.loc[:, ["position_label", "time_hours", spec["metric_name"]]]
        .copy()
    )
    for _, position_trace in per_position.groupby("position_label", sort=True):
        position_trace = position_trace.sort_values("time_hours")
        trace_x = position_trace["time_hours"].to_numpy(dtype=float)
        trace_y = position_trace[spec["metric_name"]].to_numpy(dtype=float)
        finite_trace = np.isfinite(trace_x) & np.isfinite(trace_y)
        if finite_trace.sum() < 2:
            continue
        plotted_trace = trace_y[finite_trace]
        ax.plot(
            trace_x[finite_trace],
            plotted_trace,
            color="0.75",
            linewidth=0.8,
            alpha=0.35,
            zorder=1,
        )
    summary = aggregate_mean_trace(plot_df, spec["metric_name"], "YFP")
    x = summary["time_hours"].to_numpy(dtype=float)
    y = summary["mean"].to_numpy(dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    ax.plot(x, y, color=REPORTER_COLORS["YFP"], linewidth=3.2, zorder=3)
    if finite.any():
        ax.set_ylim(
            focus_ylim_from_arrays(
                [plot_df[spec["metric_name"]].to_numpy(dtype=float), y[finite]],
                include_zero=bool(spec.get("include_zero", True)),
                padding_fraction=float(spec.get("padding_fraction", 0.10)),
            )
        )
    ax.set_title(spec["metric_title"], fontsize=10.0)
    ax.set_xlabel("Time (hours)")
    set_display_time_axis(ax, "x")
    ax.set_ylabel(spec["ylabel"])
    ax.grid(alpha=0.18)

for ax, spec in zip(axes[1], raw_yfp_specs):
    excluded = {str(position_label) for position_label in spec.get("exclude_positions", ())}
    plot_df = population_metrics.loc[
        (population_metrics["reporter"] == "YFP")
        & (~population_metrics["position_label"].isin(excluded))
    ].copy()
    per_position = (
        plot_df.loc[:, ["position_label", "time_hours", spec["metric_name"]]]
        .copy()
    )
    normalized_arrays = []
    for _, position_trace in per_position.groupby("position_label", sort=True):
        position_trace = position_trace.sort_values("time_hours")
        trace_x = position_trace["time_hours"].to_numpy(dtype=float)
        trace_y = position_trace[spec["metric_name"]].to_numpy(dtype=float)
        normalized_trace, scale_factor = normalize_raw_trace_for_display(
            trace_y,
            mode=str(spec.get("normalization_mode", "baseline_peak")),
            smooth_sigma=4.0,
            early_n=8,
        )
        max_scale_factor = float(spec.get("normalized_max_scale_factor", float("inf")))
        if np.isfinite(scale_factor) and scale_factor > max_scale_factor:
            continue
        finite_trace = np.isfinite(trace_x) & np.isfinite(normalized_trace)
        if finite_trace.sum() < 2:
            continue
        normalized_arrays.append(normalized_trace[finite_trace])
        ax.plot(
            trace_x[finite_trace],
            normalized_trace[finite_trace],
            color="0.75",
            linewidth=0.8,
            alpha=0.35,
            zorder=1,
        )
    aggregate_summary = aggregate_mean_trace(plot_df, spec["metric_name"], "YFP")
    x = aggregate_summary["time_hours"].to_numpy(dtype=float)
    y, _ = normalize_raw_trace_for_display(
        aggregate_summary["mean"].to_numpy(dtype=float),
        mode=str(spec.get("normalization_mode", "baseline_peak")),
        smooth_sigma=4.0,
        early_n=8,
    )
    finite = np.isfinite(x) & np.isfinite(y)
    if not normalized_arrays or finite.sum() < 2:
        continue
    ax.plot(x[finite], y[finite], color=REPORTER_COLORS["YFP"], linewidth=3.2, zorder=3)
    ax.set_ylim(*tuple(spec.get("normalized_ylim", (-0.5, 1.5))))
    ax.set_title(f"{spec['metric_title']} | within-cyst normalized", fontsize=10.0)
    ax.set_xlabel("Time (hours)")
    set_display_time_axis(ax, "x")
    ax.set_ylabel(str(spec.get("normalized_ylabel", "Normalized value")))
    ax.grid(alpha=0.18)

fig.suptitle("Aggregate YFP activity in clock time", fontsize=11.6)
fig.savefig(raw_yfp_clock_time_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", raw_yfp_clock_time_path)


#### Individual-Cyst YFP Traces In Clock Time

The aggregate summaries above show what the population mean looks like.

The next question is whether individual cysts show more switch-like `YFP` traces, but at different onset times.

In the plots below, the displayed traces are raw unsmoothed clock-time measurements after per-cyst normalization.

For Plot 4, each positive-fraction trace is min-max normalized to `[0, 1]` using that cyst's raw minimum and maximum. For the intensity panels, baseline and peak are estimated from a Gaussian-smoothed copy of the trace (`sigma = 4` frames), then the raw trace is rescaled as `(raw - baseline) / (peak - baseline)` with no clipping. Plots 4-6 are currently sorted by rescaling factor rather than half-max time so the trace-quality issue is visible directly.


##### Plot 4. Min-Max-Normalized Individual YFP Positive-Fraction Traces Sorted By Rescaling Factor

These are raw clock-time `YFP` positive-fraction traces, one cyst per panel, min-max normalized within cyst to `[0, 1]` and sorted by rescaling factor (`1 / (max - min)`), with the largest rescaling factors shown first.


In [ ]:
render_individual_yfp_trace_grid(
    metric_name="positive_fraction_sigma3",
    metric_title="Min-max-normalized individual YFP positive-fraction traces in clock time",
    ylabel="Normalized YFP positive fraction",
    figure_path=FIGURE_DIR / "05d_yfp_positive_fraction_individual_traces_clock_time.png",
    include_zero=True,
    normalize_by_amplitude=True,
    normalization_mode="minmax",
    y_limits_override=(-0.05, 1.05),
    n_cols=6,
    panel_width=1.45,
    panel_height=0.88,
    use_panel_legends=True,
    show_halfmax_in_panel_label=False,
    show_scale_in_panel_label=True,
    show_figure_title=False,
    max_scale_factor=None,
    sort_by="scale_factor",
    sort_descending=True,
    line_width=1.4,
    label_fontsize=4.8,
    tick_fontsize=4.8,
)


##### Plot 5. Amplitude-Normalized Individual YFP Positive-Region Mean-Intensity Traces Sorted By Rescaling Factor

These are raw clock-time `YFP` positive-region mean-intensity traces, one cyst per panel, amplitude-normalized within cyst and sorted by rescaling factor (`1 / (peak - baseline)`), with the largest rescaling factors shown first. `Pos42` is omitted here because it is an extreme outlier that compresses the scale for the remaining cysts. For readability, this montage uses a fixed y-range of `[-0.5, 1.5]` rather than allowing a few negative-tail traces to determine the global axis limits. No scale-based cutoff is applied in this montage.


In [ ]:
render_individual_yfp_trace_grid(
    metric_name="positive_mean_intensity_sigma3_z",
    metric_title="Amplitude-normalized individual YFP positive-region mean-intensity traces in clock time (Pos42 omitted)",
    ylabel="Normalized YFP positive-region mean intensity",
    figure_path=FIGURE_DIR / "05d_yfp_positive_mean_individual_traces_clock_time.png",
    include_zero=True,
    exclude_positions=("Pos42",),
    normalize_by_amplitude=True,
    y_limits_override=(-0.5, 1.5),
    n_cols=6,
    panel_width=1.45,
    panel_height=0.88,
    use_panel_legends=True,
    show_halfmax_in_panel_label=False,
    show_scale_in_panel_label=True,
    show_figure_title=False,
    max_scale_factor=None,
    sort_by="scale_factor",
    sort_descending=True,
    line_width=1.4,
    label_fontsize=4.8,
    tick_fontsize=4.8,
)


##### Plot 6. Amplitude-Normalized Individual YFP Whole-Cyst Mean-Intensity Traces Sorted By Rescaling Factor

These are raw clock-time `YFP` whole-cyst mean-intensity traces, one cyst per panel, amplitude-normalized within cyst and sorted by rescaling factor (`1 / (peak - baseline)`), with the largest rescaling factors shown first. This montage uses the same compact layout and fixed y-range as Plot 5 so the shapes are easier to compare directly.


In [ ]:
render_individual_yfp_trace_grid(
    metric_name="organoid_mean_intensity_z",
    metric_title="Amplitude-normalized individual YFP whole-cyst mean-intensity traces in clock time",
    ylabel="Normalized YFP whole-cyst mean intensity",
    figure_path=FIGURE_DIR / "05d_yfp_whole_cyst_individual_traces_clock_time.png",
    include_zero=True,
    normalize_by_amplitude=True,
    y_limits_override=(-0.5, 1.5),
    n_cols=6,
    panel_width=1.45,
    panel_height=0.88,
    use_panel_legends=True,
    show_halfmax_in_panel_label=False,
    show_scale_in_panel_label=True,
    show_figure_title=False,
    sort_by="scale_factor",
    sort_descending=True,
    line_width=1.4,
    label_fontsize=4.8,
    tick_fontsize=4.8,
)


#### Debugging Linear Versus Switch-Like BMP4 Trace Shapes

These cells are exploratory and are meant to help audit the montage above, not to make a final mechanistic claim.

To keep the comparison fair and avoid fitting a logistic specifically, the trace-shape comparison below uses:

- a 2-parameter linear model
- a 2-parameter switch-like ramp model with a lower plateau, rising segment, and upper plateau

The ramp model is not a mechanistic model of feedback. It is just a simple nonlinear alternative to a straight line.

To reduce overfitting, the comparison is based on blocked cross-validated RMSE on the normalized raw traces rather than just in-sample residuals.


In [ ]:
shape_comparison_specs = [
    {
        "metric_name": "positive_fraction_sigma3",
        "metric_label": "BMP4 positive fraction",
        "normalization_mode": "minmax",
        "exclude_positions": (),
        "y_limits": (0.0, 1.0),
    },
    {
        "metric_name": "positive_mean_intensity_sigma3_z",
        "metric_label": "BMP4 positive-region mean intensity",
        "normalization_mode": "baseline_peak",
        "exclude_positions": ("Pos42",),
        "y_limits": (-0.5, 1.5),
    },
    {
        "metric_name": "organoid_mean_intensity_z",
        "metric_label": "BMP4 whole-cyst mean intensity",
        "normalization_mode": "baseline_peak",
        "exclude_positions": (),
        "y_limits": (-0.5, 1.5),
    },
]

ramp_candidate_cache = {}


def normalize_trace_for_shape_comparison(
    values: np.ndarray,
    mode: str,
    smooth_sigma: float = 4.0,
    early_n: int = 8,
) -> tuple[np.ndarray, float]:
    arr = np.asarray(values, dtype=float)
    if mode == "minmax":
        finite_arr = arr[np.isfinite(arr)]
        if finite_arr.size < 2:
            return np.full_like(arr, np.nan, dtype=float), float("nan")
        baseline = float(np.nanmin(finite_arr))
        peak = float(np.nanmax(finite_arr))
        amplitude = peak - baseline
        if not np.isfinite(amplitude) or amplitude <= 0:
            return np.full_like(arr, np.nan, dtype=float), float("nan")
        return (arr - baseline) / amplitude, float(1.0 / amplitude)

    smoothed = gaussian_smooth(arr, smooth_sigma)
    finite_smoothed = smoothed[np.isfinite(smoothed)]
    if finite_smoothed.size < max(early_n, 4):
        return np.full_like(arr, np.nan, dtype=float), float("nan")
    baseline = float(np.nanmedian(smoothed[:early_n]))
    peak = float(np.nanmax(smoothed))
    amplitude = peak - baseline
    if not np.isfinite(amplitude) or amplitude <= 0:
        return np.full_like(arr, np.nan, dtype=float), float("nan")
    return (arr - baseline) / amplitude, float(1.0 / amplitude)


def ramp_prediction(time_hours: np.ndarray, onset_hours: float, offset_hours: float) -> np.ndarray:
    t = np.asarray(time_hours, dtype=float)
    if not np.isfinite(onset_hours) or not np.isfinite(offset_hours) or offset_hours <= onset_hours:
        return np.full_like(t, np.nan, dtype=float)
    return np.clip((t - onset_hours) / (offset_hours - onset_hours), 0.0, 1.0)


def fit_linear_trace(time_hours: np.ndarray, values: np.ndarray) -> dict | None:
    t = np.asarray(time_hours, dtype=float)
    y = np.asarray(values, dtype=float)
    finite = np.isfinite(t) & np.isfinite(y)
    t = t[finite]
    y = y[finite]
    if t.size < 3:
        return None
    design = np.column_stack([np.ones_like(t), t])
    beta, _, _, _ = np.linalg.lstsq(design, y, rcond=None)
    yhat = design @ beta
    rss = float(np.sum((y - yhat) ** 2))
    return {
        "intercept": float(beta[0]),
        "slope": float(beta[1]),
        "rss": rss,
    }


def predict_linear_trace(fit_info: dict, time_hours: np.ndarray) -> np.ndarray:
    t = np.asarray(time_hours, dtype=float)
    return fit_info["intercept"] + fit_info["slope"] * t


def fit_switch_ramp_trace(
    time_hours: np.ndarray,
    values: np.ndarray,
    min_span_hours: float = 6.0,
) -> dict | None:
    t = np.asarray(time_hours, dtype=float)
    y = np.asarray(values, dtype=float)
    finite = np.isfinite(t) & np.isfinite(y)
    t = t[finite]
    y = y[finite]
    if t.size < 4:
        return None
    unique_times = np.unique(t)
    if unique_times.size < 2:
        return None

    cache_key = (tuple(np.round(unique_times, 6).tolist()), float(min_span_hours))
    candidate_pairs = ramp_candidate_cache.get(cache_key)
    if candidate_pairs is None:
        candidate_pairs = [
            (float(t_on), float(t_off))
            for i, t_on in enumerate(unique_times[:-1])
            for t_off in unique_times[i + 1 :]
            if (t_off - t_on) >= min_span_hours
        ]
        if not candidate_pairs:
            candidate_pairs = [
                (float(t_on), float(t_off))
                for i, t_on in enumerate(unique_times[:-1])
                for t_off in unique_times[i + 1 :]
            ]
        ramp_candidate_cache[cache_key] = candidate_pairs
    if not candidate_pairs:
        return None

    best = None
    for onset_hours, offset_hours in candidate_pairs:
        yhat = ramp_prediction(t, onset_hours, offset_hours)
        rss = float(np.sum((y - yhat) ** 2))
        if best is None or rss < best["rss"]:
            best = {
                "onset_hours": onset_hours,
                "offset_hours": offset_hours,
                "rss": rss,
            }
    return best


def contiguous_block_folds(n_points: int, max_folds: int = 5, min_holdout_points: int = 3) -> list[tuple[int, int]]:
    n_folds = min(max_folds, max(2, n_points // min_holdout_points))
    while n_folds >= 2:
        edges = np.linspace(0, n_points, n_folds + 1, dtype=int)
        sizes = np.diff(edges)
        if np.all(sizes >= min_holdout_points):
            return [(int(edges[i]), int(edges[i + 1])) for i in range(n_folds)]
        n_folds -= 1
    return []


def blocked_cv_rmse(
    time_hours: np.ndarray,
    values: np.ndarray,
    model_kind: str,
    min_train_points: int = 8,
) -> float:
    t = np.asarray(time_hours, dtype=float)
    y = np.asarray(values, dtype=float)
    finite = np.isfinite(t) & np.isfinite(y)
    t = t[finite]
    y = y[finite]
    if t.size < (min_train_points + 3):
        return float("nan")

    rmses = []
    for start_idx, stop_idx in contiguous_block_folds(len(t), max_folds=5, min_holdout_points=3):
        test_mask = np.zeros(len(t), dtype=bool)
        test_mask[start_idx:stop_idx] = True
        train_mask = ~test_mask
        if train_mask.sum() < min_train_points or test_mask.sum() < 3:
            continue
        if model_kind == "linear":
            fit_info = fit_linear_trace(t[train_mask], y[train_mask])
            if fit_info is None:
                continue
            predicted = predict_linear_trace(fit_info, t[test_mask])
        elif model_kind == "ramp":
            fit_info = fit_switch_ramp_trace(t[train_mask], y[train_mask], min_span_hours=6.0)
            if fit_info is None:
                continue
            predicted = ramp_prediction(t[test_mask], fit_info["onset_hours"], fit_info["offset_hours"])
        else:
            raise ValueError(f"Unknown model kind: {model_kind}")
        finite_pred = np.isfinite(predicted) & np.isfinite(y[test_mask])
        if finite_pred.sum() < 2:
            continue
        rmse = float(np.sqrt(np.mean((y[test_mask][finite_pred] - predicted[finite_pred]) ** 2)))
        rmses.append(rmse)
    return float(np.mean(rmses)) if rmses else float("nan")


shape_rows = []
normalized_trace_lookup = {}
for spec in shape_comparison_specs:
    excluded = {str(position_label) for position_label in spec.get("exclude_positions", ())}
    filtered_df = population_metrics.loc[
        (population_metrics["reporter"] == "YFP")
        & (~population_metrics["position_label"].isin(excluded))
    ].copy()

    for position_label, subset in (
        filtered_df.loc[:, ["position_label", "time_hours", spec["metric_name"]]]
        .groupby("position_label", sort=True)
    ):
        ordered = subset.sort_values("time_hours")
        raw_values = ordered[spec["metric_name"]].to_numpy(dtype=float)
        normalized_values, scale_factor = normalize_trace_for_shape_comparison(
            raw_values,
            mode=str(spec["normalization_mode"]),
            smooth_sigma=4.0,
            early_n=8,
        )
        time_hours = ordered["time_hours"].to_numpy(dtype=float)
        finite = np.isfinite(time_hours) & np.isfinite(normalized_values)
        if finite.sum() < 12:
            continue
        valid_time = time_hours[finite]
        valid_values = normalized_values[finite]

        linear_fit = fit_linear_trace(valid_time, valid_values)
        ramp_fit = fit_switch_ramp_trace(valid_time, valid_values, min_span_hours=6.0)
        if linear_fit is None or ramp_fit is None:
            continue

        linear_cv_rmse = blocked_cv_rmse(valid_time, valid_values, "linear")
        ramp_cv_rmse = blocked_cv_rmse(valid_time, valid_values, "ramp")
        cv_improvement_factor = (
            float(linear_cv_rmse / ramp_cv_rmse)
            if np.isfinite(linear_cv_rmse) and np.isfinite(ramp_cv_rmse) and ramp_cv_rmse > 0
            else float("nan")
        )
        cv_delta_rmse = (
            float(linear_cv_rmse - ramp_cv_rmse)
            if np.isfinite(linear_cv_rmse) and np.isfinite(ramp_cv_rmse)
            else float("nan")
        )
        if np.isfinite(cv_improvement_factor):
            if cv_improvement_factor > 1.10:
                cv_call = "ramp-favored"
            elif cv_improvement_factor < (1.0 / 1.10):
                cv_call = "linear-favored"
            else:
                cv_call = "ambiguous"
        else:
            cv_call = "unscored"

        halfmax_time = halfmax_time_for_position(
            filtered_df,
            spec["metric_name"],
            position_label,
            "YFP",
            smooth_sigma=4.0,
        )

        shape_rows.append(
            {
                "metric_name": spec["metric_name"],
                "metric_label": spec["metric_label"],
                "position_label": position_label,
                "normalization_mode": spec["normalization_mode"],
                "n_points": int(valid_time.size),
                "scale_factor": float(scale_factor),
                "halfmax_time_hours": float(halfmax_time),
                "line_rss": float(linear_fit["rss"]),
                "line_intercept": float(linear_fit["intercept"]),
                "line_slope": float(linear_fit["slope"]),
                "ramp_rss": float(ramp_fit["rss"]),
                "ramp_onset_hours": float(ramp_fit["onset_hours"]),
                "ramp_offset_hours": float(ramp_fit["offset_hours"]),
                "line_cv_rmse": float(linear_cv_rmse),
                "ramp_cv_rmse": float(ramp_cv_rmse),
                "cv_delta_rmse": float(cv_delta_rmse),
                "cv_improvement_factor": float(cv_improvement_factor),
                "cv_call": cv_call,
            }
        )
        normalized_trace_lookup[(spec["metric_name"], str(position_label))] = {
            "time_hours": valid_time,
            "values": valid_values,
            "y_limits": spec["y_limits"],
        }

shape_comparison_df = pd.DataFrame(shape_rows)
shape_comparison_path = TABLE_DIR / "05d_bmp4_trace_shape_linear_vs_ramp.tsv"
shape_comparison_df.to_csv(shape_comparison_path, sep="\t", index=False)

shape_summary_df = (
    shape_comparison_df.groupby("metric_label", as_index=False)
    .agg(
        n_traces=("position_label", "size"),
        median_scale_factor=("scale_factor", "median"),
        median_line_cv_rmse=("line_cv_rmse", "median"),
        median_ramp_cv_rmse=("ramp_cv_rmse", "median"),
        median_cv_improvement_factor=("cv_improvement_factor", "median"),
        fraction_ramp_favored=("cv_call", lambda values: float(np.mean(pd.Series(values) == "ramp-favored"))),
        fraction_ambiguous=("cv_call", lambda values: float(np.mean(pd.Series(values) == "ambiguous"))),
        fraction_linear_favored=("cv_call", lambda values: float(np.mean(pd.Series(values) == "linear-favored"))),
    )
    .sort_values("metric_label")
)

display(Markdown("**Per-metric summary**"))
display(shape_summary_df.round(3))

display(Markdown("**Most ramp-favored traces by blocked CV**"))
display(
    shape_comparison_df.sort_values(["metric_label", "cv_improvement_factor"], ascending=[True, False])
    .groupby("metric_label", as_index=False)
    .head(6)
    .round(3)
)

display(Markdown("**Most linear-favored traces by blocked CV**"))
display(
    shape_comparison_df.sort_values(["metric_label", "cv_improvement_factor"], ascending=[True, True])
    .groupby("metric_label", as_index=False)
    .head(6)
    .round(3)
)

print("Wrote table:", shape_comparison_path)


##### Debug A. Cross-Validated Linear-Versus-Ramp Comparison

The diagonal in the top row marks equal blocked-CV RMSE. Points below the diagonal favor the switch-like ramp model.

The bottom row shows how much that model advantage tracks the rescaling factor used to normalize each trace.


In [ ]:
comparison_figure_path = FIGURE_DIR / "05d_bmp4_trace_shape_cv_comparison.png"
fig, axes = plt.subplots(2, len(shape_comparison_specs), figsize=(5.1 * len(shape_comparison_specs), 7.8), constrained_layout=True)

for col_idx, spec in enumerate(shape_comparison_specs):
    metric_df = shape_comparison_df.loc[shape_comparison_df["metric_name"] == spec["metric_name"]].copy()
    top_ax = axes[0, col_idx]
    bottom_ax = axes[1, col_idx]

    finite_cv = metric_df.loc[
        np.isfinite(metric_df["line_cv_rmse"].to_numpy(dtype=float))
        & np.isfinite(metric_df["ramp_cv_rmse"].to_numpy(dtype=float))
    ].copy()
    if len(finite_cv):
        top_ax.scatter(
            finite_cv["line_cv_rmse"],
            finite_cv["ramp_cv_rmse"],
            c=np.log10(np.clip(finite_cv["scale_factor"].to_numpy(dtype=float), 1e-6, None)),
            cmap="viridis",
            s=24,
            alpha=0.9,
            edgecolor="none",
        )
        xy_min = float(np.nanmin([finite_cv["line_cv_rmse"].min(), finite_cv["ramp_cv_rmse"].min()]))
        xy_max = float(np.nanmax([finite_cv["line_cv_rmse"].max(), finite_cv["ramp_cv_rmse"].max()]))
        pad = 0.06 * (xy_max - xy_min if xy_max > xy_min else 1.0)
        top_ax.plot([xy_min - pad, xy_max + pad], [xy_min - pad, xy_max + pad], color="0.55", linestyle="--", linewidth=1.0)
        top_ax.set_xlim(xy_min - pad, xy_max + pad)
        top_ax.set_ylim(xy_min - pad, xy_max + pad)
    top_ax.set_title(spec["metric_label"], fontsize=10.0)
    top_ax.set_xlabel("Linear blocked-CV RMSE")
    top_ax.set_ylabel("Ramp blocked-CV RMSE")
    top_ax.grid(alpha=0.18)
    if len(metric_df):
        top_ax.text(
            0.03,
            0.97,
            "\n".join(
                [
                    f"n = {len(metric_df)}",
                    f"ramp-favored = {(metric_df['cv_call'] == 'ramp-favored').mean():.2f}",
                    f"linear-favored = {(metric_df['cv_call'] == 'linear-favored').mean():.2f}",
                ]
            ),
            transform=top_ax.transAxes,
            ha="left",
            va="top",
            fontsize=8.0,
            bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "0.8", "alpha": 0.9},
        )

    finite_scale = metric_df.loc[
        np.isfinite(metric_df["scale_factor"].to_numpy(dtype=float))
        & np.isfinite(metric_df["cv_improvement_factor"].to_numpy(dtype=float))
        & (metric_df["scale_factor"].to_numpy(dtype=float) > 0)
    ].copy()
    if len(finite_scale):
        bottom_ax.scatter(
            finite_scale["scale_factor"],
            finite_scale["cv_improvement_factor"],
            color=REPORTER_COLORS["YFP"],
            s=22,
            alpha=0.8,
            edgecolor="none",
        )
        bottom_ax.set_xscale("log")
    bottom_ax.axhline(1.0, color="0.55", linestyle="--", linewidth=1.0)
    bottom_ax.set_xlabel("Rescaling factor")
    bottom_ax.set_ylabel("Linear / ramp blocked-CV RMSE")
    bottom_ax.grid(alpha=0.18)

fig.suptitle("BMP4 trace-shape comparison: linear versus switch-like ramp", fontsize=11.6)
fig.savefig(comparison_figure_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", comparison_figure_path)


##### Debug B. Representative Trace Fits

These panels are meant to be visually debuggable. The raw normalized trace is shown in black, the linear fit in blue, and the switch-like ramp fit in red. Red dashed lines mark the fitted ramp onset and offset.


In [ ]:
representative_fit_path = FIGURE_DIR / "05d_bmp4_trace_shape_representative_fits.png"

def choose_representatives(metric_df: pd.DataFrame) -> list[pd.Series]:
    representatives = []
    used_positions = set()

    candidate_frames = [
        metric_df.sort_values("cv_improvement_factor", ascending=False),
        metric_df.assign(tie_distance=(metric_df["cv_improvement_factor"] - 1.0).abs()).sort_values("tie_distance", ascending=True),
        metric_df.sort_values("cv_improvement_factor", ascending=True),
        metric_df.sort_values("scale_factor", ascending=False),
    ]

    for candidate_df in candidate_frames:
        chosen = None
        for _, row in candidate_df.iterrows():
            if row["position_label"] in used_positions:
                continue
            chosen = row
            break
        if chosen is None:
            chosen = candidate_df.iloc[0]
        representatives.append(chosen)
        used_positions.add(chosen["position_label"])
    return representatives


fig, axes = plt.subplots(len(shape_comparison_specs), 4, figsize=(16.0, 3.8 * len(shape_comparison_specs)), constrained_layout=True)
column_titles = ["Most ramp-favored", "Near tie", "Most linear-favored", "Largest rescale"]
for ax, title in zip(axes[0], column_titles):
    ax.set_title(title, fontsize=10.0)

for row_idx, spec in enumerate(shape_comparison_specs):
    metric_df = shape_comparison_df.loc[shape_comparison_df["metric_name"] == spec["metric_name"]].copy()
    representatives = choose_representatives(metric_df)
    for col_idx, (_, row) in enumerate(pd.DataFrame(representatives).iterrows()):
        ax = axes[row_idx, col_idx]
        trace_info = normalized_trace_lookup[(spec["metric_name"], row["position_label"])]
        time_hours = trace_info["time_hours"]
        values = trace_info["values"]
        line_prediction = row["line_intercept"] + row["line_slope"] * time_hours
        ramp_values = ramp_prediction(time_hours, row["ramp_onset_hours"], row["ramp_offset_hours"])

        ax.plot(time_hours, values, color="0.15", linewidth=1.8, label="normalized trace")
        ax.plot(time_hours, line_prediction, color=FIT_COLORS["linear"], linewidth=2.0, label="linear")
        ax.plot(time_hours, ramp_values, color=FIT_COLORS["logistic"], linewidth=2.0, label="ramp")
        ax.axvline(row["ramp_onset_hours"], color=FIT_COLORS["logistic"], linestyle="--", linewidth=0.9, alpha=0.75)
        ax.axvline(row["ramp_offset_hours"], color=FIT_COLORS["logistic"], linestyle="--", linewidth=0.9, alpha=0.75)
        ax.set_ylim(*trace_info["y_limits"])
        ax.set_xlabel("Time (hours)")
        ax.set_ylabel(spec["metric_label"])
        ax.grid(alpha=0.18)
        ax.text(
            0.03,
            0.03,
            "\n".join(
                [
                    f"{row['position_label']} | scale x{row['scale_factor']:.1f}",
                    f"CV ratio = {row['cv_improvement_factor']:.2f}",
                    f"call = {row['cv_call']}",
                ]
            ),
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=7.7,
            bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "0.8", "alpha": 0.9},
        )
        if row_idx == 0 and col_idx == 0:
            ax.legend(loc="upper left", frameon=False, fontsize=8.0)

fig.suptitle("Representative BMP4 trace fits: linear versus switch-like ramp", fontsize=11.6)
fig.savefig(representative_fit_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", representative_fit_path)


##### Debug C. Aggregate BMP4 Mean Traces: Linear Versus Switch-Like Ramp

For the aggregate traces, the timepoints are not the unit of replication, so I do not treat a time-block CV score on the mean trace as the main inferential statistic.

Instead, the aggregate comparison below is handled more conservatively:

- the observed aggregate mean trace is normalized once as a single curve and fit with matched-complexity linear and ramp models
- uncertainty is summarized by bootstrapping positions, not timepoints

Because both fitted shapes have two free parameters after normalization, the aggregate score here is the in-sample RMSE on the normalized aggregate curve. I am treating that as descriptive/supportive rather than decisive.


In [ ]:
AGGREGATE_BOOTSTRAP_REPS = 100
AGGREGATE_BOOTSTRAP_CURVES_TO_PLOT = 15


def classify_shape_ratio(improvement_factor: float, threshold: float = 1.10) -> str:
    if not np.isfinite(improvement_factor):
        return "unscored"
    if improvement_factor > threshold:
        return "ramp-favored"
    if improvement_factor < (1.0 / threshold):
        return "linear-favored"
    return "ambiguous"


def aggregate_trace_for_shape_comparison(
    filtered_df: pd.DataFrame,
    metric_name: str,
    normalization_mode: str,
) -> dict | None:
    summary = aggregate_mean_trace(filtered_df, metric_name, "YFP")
    time_hours = summary["time_hours"].to_numpy(dtype=float)
    mean_values = summary["mean"].to_numpy(dtype=float)
    normalized_values, scale_factor = normalize_trace_for_shape_comparison(
        mean_values,
        mode=normalization_mode,
        smooth_sigma=4.0,
        early_n=8,
    )
    finite = np.isfinite(time_hours) & np.isfinite(normalized_values)
    if finite.sum() < 12:
        return None
    return {
        "time_hours": time_hours[finite],
        "values": normalized_values[finite],
        "raw_summary": summary,
        "scale_factor": float(scale_factor),
    }


def bootstrap_aggregate_shape_comparison(
    filtered_df: pd.DataFrame,
    metric_name: str,
    normalization_mode: str,
    n_bootstrap: int = 200,
    rng_seed: int = 0,
    n_curves_to_keep: int = 20,
) -> tuple[pd.DataFrame, list[dict]]:
    rng = np.random.default_rng(rng_seed)
    by_position = {
        str(position_label): subset.loc[:, ["time_hours", metric_name]].sort_values("time_hours").copy()
        for position_label, subset in filtered_df.groupby("position_label", sort=True)
    }
    positions = sorted(by_position)
    bootstrap_rows = []
    bootstrap_curves = []
    if not positions:
        return pd.DataFrame(), bootstrap_curves

    for bootstrap_idx in range(n_bootstrap):
        sampled_positions = rng.choice(positions, size=len(positions), replace=True)
        sampled_frames = []
        for draw_idx, position_label in enumerate(sampled_positions):
            frame = by_position[str(position_label)].copy()
            frame["bootstrap_draw"] = int(draw_idx)
            sampled_frames.append(frame)
        sampled_df = pd.concat(sampled_frames, ignore_index=True)
        aggregate_summary = (
            sampled_df.groupby("time_hours", as_index=False)
            .agg(mean=(metric_name, "mean"))
            .sort_values("time_hours")
            .reset_index(drop=True)
        )
        normalized_values, scale_factor = normalize_trace_for_shape_comparison(
            aggregate_summary["mean"].to_numpy(dtype=float),
            mode=normalization_mode,
            smooth_sigma=4.0,
            early_n=8,
        )
        time_hours = aggregate_summary["time_hours"].to_numpy(dtype=float)
        finite = np.isfinite(time_hours) & np.isfinite(normalized_values)
        if finite.sum() < 12:
            continue

        valid_time = time_hours[finite]
        valid_values = normalized_values[finite]
        linear_fit = fit_linear_trace(valid_time, valid_values)
        ramp_fit = fit_switch_ramp_trace(valid_time, valid_values, min_span_hours=6.0)
        if linear_fit is None or ramp_fit is None:
            continue

        line_rmse = float(np.sqrt(linear_fit["rss"] / len(valid_time)))
        ramp_rmse = float(np.sqrt(ramp_fit["rss"] / len(valid_time)))
        improvement_factor = float(line_rmse / ramp_rmse) if np.isfinite(ramp_rmse) and ramp_rmse > 0 else float("nan")

        bootstrap_rows.append(
            {
                "bootstrap_index": int(bootstrap_idx),
                "n_positions": int(len(sampled_positions)),
                "n_timepoints": int(len(valid_time)),
                "scale_factor": float(scale_factor),
                "line_rmse": line_rmse,
                "ramp_rmse": ramp_rmse,
                "improvement_factor": improvement_factor,
                "shape_call": classify_shape_ratio(improvement_factor),
                "ramp_onset_hours": float(ramp_fit["onset_hours"]),
                "ramp_offset_hours": float(ramp_fit["offset_hours"]),
            }
        )

        if len(bootstrap_curves) < n_curves_to_keep:
            bootstrap_curves.append(
                {
                    "time_hours": valid_time,
                    "values": valid_values,
                }
            )

    return pd.DataFrame(bootstrap_rows), bootstrap_curves


aggregate_fit_rows = []
aggregate_bootstrap_frames = []
aggregate_curve_lookup = {}

for spec in shape_comparison_specs:
    excluded = {str(position_label) for position_label in spec.get("exclude_positions", ())}
    filtered_df = population_metrics.loc[
        (population_metrics["reporter"] == "YFP")
        & (~population_metrics["position_label"].isin(excluded))
    ].copy()
    aggregate_trace = aggregate_trace_for_shape_comparison(
        filtered_df,
        spec["metric_name"],
        str(spec["normalization_mode"]),
    )
    if aggregate_trace is None:
        continue

    valid_time = aggregate_trace["time_hours"]
    valid_values = aggregate_trace["values"]
    linear_fit = fit_linear_trace(valid_time, valid_values)
    ramp_fit = fit_switch_ramp_trace(valid_time, valid_values, min_span_hours=6.0)
    if linear_fit is None or ramp_fit is None:
        continue

    line_prediction = predict_linear_trace(linear_fit, valid_time)
    ramp_values = ramp_prediction(valid_time, ramp_fit["onset_hours"], ramp_fit["offset_hours"])
    line_rmse = float(np.sqrt(linear_fit["rss"] / len(valid_time)))
    ramp_rmse = float(np.sqrt(ramp_fit["rss"] / len(valid_time)))
    improvement_factor = float(line_rmse / ramp_rmse) if np.isfinite(ramp_rmse) and ramp_rmse > 0 else float("nan")

    bootstrap_df, bootstrap_curves = bootstrap_aggregate_shape_comparison(
        filtered_df,
        spec["metric_name"],
        str(spec["normalization_mode"]),
        n_bootstrap=AGGREGATE_BOOTSTRAP_REPS,
        rng_seed=17 + len(aggregate_fit_rows),
        n_curves_to_keep=AGGREGATE_BOOTSTRAP_CURVES_TO_PLOT,
    )
    if len(bootstrap_df):
        bootstrap_df.insert(0, "metric_name", spec["metric_name"])
        bootstrap_df.insert(1, "metric_label", spec["metric_label"])
        aggregate_bootstrap_frames.append(bootstrap_df)
        bootstrap_median_ratio = float(np.nanmedian(bootstrap_df["improvement_factor"]))
        bootstrap_low_ratio = float(np.nanquantile(bootstrap_df["improvement_factor"], 0.025))
        bootstrap_high_ratio = float(np.nanquantile(bootstrap_df["improvement_factor"], 0.975))
        bootstrap_fraction_ramp = float(np.mean(bootstrap_df["improvement_factor"] > 1.0))
    else:
        bootstrap_median_ratio = float("nan")
        bootstrap_low_ratio = float("nan")
        bootstrap_high_ratio = float("nan")
        bootstrap_fraction_ramp = float("nan")

    aggregate_fit_rows.append(
        {
            "metric_name": spec["metric_name"],
            "metric_label": spec["metric_label"],
            "n_positions": int(filtered_df["position_label"].nunique()),
            "n_timepoints": int(len(valid_time)),
            "normalization_mode": spec["normalization_mode"],
            "scale_factor": float(aggregate_trace["scale_factor"]),
            "line_rmse": line_rmse,
            "ramp_rmse": ramp_rmse,
            "improvement_factor": improvement_factor,
            "shape_call": classify_shape_ratio(improvement_factor),
            "line_intercept": float(linear_fit["intercept"]),
            "line_slope": float(linear_fit["slope"]),
            "ramp_onset_hours": float(ramp_fit["onset_hours"]),
            "ramp_offset_hours": float(ramp_fit["offset_hours"]),
            "bootstrap_median_ratio": bootstrap_median_ratio,
            "bootstrap_ratio_q025": bootstrap_low_ratio,
            "bootstrap_ratio_q975": bootstrap_high_ratio,
            "bootstrap_fraction_ratio_gt1": bootstrap_fraction_ramp,
        }
    )
    aggregate_curve_lookup[spec["metric_name"]] = {
        "time_hours": valid_time,
        "values": valid_values,
        "line_prediction": line_prediction,
        "ramp_prediction": ramp_values,
        "bootstrap_curves": bootstrap_curves,
        "y_limits": spec["y_limits"],
    }

aggregate_fit_df = pd.DataFrame(aggregate_fit_rows).sort_values("metric_label").reset_index(drop=True)
aggregate_bootstrap_df = (
    pd.concat(aggregate_bootstrap_frames, ignore_index=True)
    if aggregate_bootstrap_frames
    else pd.DataFrame()
)

aggregate_fit_path = TABLE_DIR / "05d_bmp4_aggregate_shape_linear_vs_ramp.tsv"
aggregate_fit_df.to_csv(aggregate_fit_path, sep="\t", index=False)

aggregate_bootstrap_path = TABLE_DIR / "05d_bmp4_aggregate_shape_bootstrap.tsv"
aggregate_bootstrap_df.to_csv(aggregate_bootstrap_path, sep="\t", index=False)

display(Markdown("**Aggregate mean-trace fit summary**"))
display(aggregate_fit_df.round(3))
if len(aggregate_bootstrap_df):
    display(Markdown("**Aggregate bootstrap summary**"))
    display(
        aggregate_bootstrap_df.groupby("metric_label", as_index=False)
        .agg(
            n_bootstrap=("bootstrap_index", "size"),
            median_ratio=("improvement_factor", "median"),
            ratio_q025=("improvement_factor", lambda values: float(np.nanquantile(values, 0.025))),
            ratio_q975=("improvement_factor", lambda values: float(np.nanquantile(values, 0.975))),
            fraction_ratio_gt1=("improvement_factor", lambda values: float(np.mean(pd.Series(values) > 1.0))),
        )
        .round(3)
    )

print("Wrote table:", aggregate_fit_path)
print("Wrote table:", aggregate_bootstrap_path)


##### Debug D. Aggregate BMP4 Mean-Trace Fits And Position Bootstrap

Top row: normalized aggregate mean traces with matched-complexity linear and ramp fits. Faint gray curves are bootstrap aggregate means from resampled positions.

Bottom row: bootstrap distributions of the aggregate line-versus-ramp RMSE ratio. Values above `1` favor the switch-like ramp.


In [ ]:
aggregate_figure_path = FIGURE_DIR / "05d_bmp4_aggregate_shape_comparison.png"
fig, axes = plt.subplots(2, len(shape_comparison_specs), figsize=(5.2 * len(shape_comparison_specs), 7.8), constrained_layout=True)

for col_idx, spec in enumerate(shape_comparison_specs):
    top_ax = axes[0, col_idx]
    bottom_ax = axes[1, col_idx]

    fit_row = aggregate_fit_df.loc[aggregate_fit_df["metric_name"] == spec["metric_name"]]
    if len(fit_row):
        fit_row = fit_row.iloc[0]
        curve_info = aggregate_curve_lookup[spec["metric_name"]]
        for bootstrap_curve in curve_info["bootstrap_curves"]:
            top_ax.plot(
                bootstrap_curve["time_hours"],
                bootstrap_curve["values"],
                color="0.80",
                linewidth=0.8,
                alpha=0.45,
            )
        top_ax.plot(
            curve_info["time_hours"],
            curve_info["values"],
            color="0.12",
            linewidth=2.4,
            label="aggregate mean",
        )
        top_ax.plot(
            curve_info["time_hours"],
            curve_info["line_prediction"],
            color=FIT_COLORS["linear"],
            linewidth=2.0,
            label="linear",
        )
        top_ax.plot(
            curve_info["time_hours"],
            curve_info["ramp_prediction"],
            color=FIT_COLORS["logistic"],
            linewidth=2.0,
            label="switch-like ramp",
        )
        top_ax.set_ylim(*curve_info["y_limits"])
        top_ax.text(
            0.03,
            0.03,
            "\n".join(
                [
                    f"ratio = {fit_row['improvement_factor']:.2f}",
                    f"line RMSE = {fit_row['line_rmse']:.3f}",
                    f"ramp RMSE = {fit_row['ramp_rmse']:.3f}",
                    f"bootstrap median = {fit_row['bootstrap_median_ratio']:.2f}",
                    f"95% interval = [{fit_row['bootstrap_ratio_q025']:.2f}, {fit_row['bootstrap_ratio_q975']:.2f}]",
                ]
            ),
            transform=top_ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=7.5,
            bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "0.8", "alpha": 0.92},
        )
    top_ax.set_title(spec["metric_label"], fontsize=10.0)
    top_ax.set_xlabel("Clock time (hours)")
    top_ax.set_ylabel("Normalized aggregate BMP4")
    top_ax.grid(alpha=0.18)
    set_display_time_axis(top_ax, axis="x", crowded=True)
    if col_idx == 0:
        top_ax.legend(loc="upper left", frameon=False, fontsize=8.0)

    metric_bootstrap_df = aggregate_bootstrap_df.loc[aggregate_bootstrap_df["metric_name"] == spec["metric_name"]].copy()
    if len(metric_bootstrap_df):
        bottom_ax.hist(
            metric_bootstrap_df["improvement_factor"].to_numpy(dtype=float),
            bins=18,
            color=REPORTER_COLORS["YFP"],
            alpha=0.82,
            edgecolor="white",
        )
        observed_ratio = float(fit_row["improvement_factor"]) if len(fit_row) else float("nan")
        if np.isfinite(observed_ratio):
            bottom_ax.axvline(observed_ratio, color="0.10", linewidth=2.0, label="observed")
    bottom_ax.axvline(1.0, color="0.55", linestyle="--", linewidth=1.0, label="equal RMSE")
    bottom_ax.set_xlabel("Linear / ramp RMSE")
    bottom_ax.set_ylabel("Bootstrap count")
    bottom_ax.grid(alpha=0.18)

fig.suptitle("Aggregate BMP4 mean traces: linear versus switch-like ramp", fontsize=11.6)
fig.savefig(aggregate_figure_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", aggregate_figure_path)


##### Debug E. Primary-Metric Linear Null For Individual BMP4 Positive-Fraction Traces

For the primary `BMP4 positive fraction` metric, the next question is whether the observed line-versus-ramp advantage is stronger than what we would expect from a genuinely linear trace with the same sampling pattern and roughly the same residual structure.

To keep this null reasonably fair without making it too parametric:

- each simulated trace keeps the observed timepoints and fitted linear trend
- residuals are resampled in short contiguous blocks from that trace's own linear residuals
- simulated traces are not clipped, to avoid introducing artificial plateaus

Because the linear and ramp models have the same number of fitted parameters after normalization, this null uses the line-versus-ramp in-sample RMSE ratio rather than rerunning blocked CV inside every synthetic trace. I think that is a reasonable compromise here: still matched-complexity, but far more tractable for iterative notebook work.


In [ ]:
LINEAR_NULL_REPS = 20
NULL_BLOCK_LENGTH = 3
PRIMARY_TRACE_METRIC = "positive_fraction_sigma3"


def moving_block_resample(residuals: np.ndarray, block_length: int, rng: np.random.Generator) -> np.ndarray:
    arr = np.asarray(residuals, dtype=float)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.full_like(arr, np.nan, dtype=float)
    centered = finite - float(np.nanmean(finite))
    n = centered.size
    if n <= block_length:
        return rng.choice(centered, size=n, replace=True)

    blocks = []
    total = 0
    max_start = max(1, n - block_length + 1)
    while total < n:
        start_idx = int(rng.integers(0, max_start))
        block = centered[start_idx : start_idx + block_length]
        blocks.append(block)
        total += len(block)
    return np.concatenate(blocks)[:n]


primary_metric_df = (
    shape_comparison_df.loc[shape_comparison_df["metric_name"] == PRIMARY_TRACE_METRIC]
    .copy()
    .sort_values("position_label")
    .reset_index(drop=True)
)
primary_metric_df["line_rmse_in_sample"] = np.sqrt(primary_metric_df["line_rss"] / primary_metric_df["n_points"])
primary_metric_df["ramp_rmse_in_sample"] = np.sqrt(primary_metric_df["ramp_rss"] / primary_metric_df["n_points"])
primary_metric_df["improvement_factor_in_sample"] = (
    primary_metric_df["line_rmse_in_sample"] / primary_metric_df["ramp_rmse_in_sample"]
)
primary_metric_df["shape_call_in_sample"] = primary_metric_df["improvement_factor_in_sample"].map(classify_shape_ratio)

rng = np.random.default_rng(20260331)
null_trace_rows = []
for simulation_index in range(LINEAR_NULL_REPS):
    for _, row in primary_metric_df.iterrows():
        trace_info = normalized_trace_lookup[(PRIMARY_TRACE_METRIC, row["position_label"])]
        time_hours = trace_info["time_hours"]
        observed_values = trace_info["values"]
        linear_prediction = row["line_intercept"] + row["line_slope"] * time_hours
        residuals = observed_values - linear_prediction
        simulated_values = linear_prediction + moving_block_resample(residuals, NULL_BLOCK_LENGTH, rng)

        simulated_linear_fit = fit_linear_trace(time_hours, simulated_values)
        simulated_ramp_fit = fit_switch_ramp_trace(time_hours, simulated_values, min_span_hours=6.0)
        if simulated_linear_fit is None or simulated_ramp_fit is None:
            continue
        line_rmse = float(np.sqrt(simulated_linear_fit["rss"] / len(time_hours)))
        ramp_rmse = float(np.sqrt(simulated_ramp_fit["rss"] / len(time_hours)))
        improvement_factor = (
            float(line_rmse / ramp_rmse)
            if np.isfinite(line_rmse) and np.isfinite(ramp_rmse) and ramp_rmse > 0
            else float("nan")
        )
        null_trace_rows.append(
            {
                "simulation_index": int(simulation_index),
                "position_label": row["position_label"],
                "scale_factor": float(row["scale_factor"]),
                "line_rmse_in_sample": float(line_rmse),
                "ramp_rmse_in_sample": float(ramp_rmse),
                "improvement_factor_in_sample": float(improvement_factor),
                "shape_call_in_sample": classify_shape_ratio(improvement_factor),
            }
        )

null_trace_df = pd.DataFrame(null_trace_rows)
null_simulation_summary_df = (
    null_trace_df.groupby("simulation_index", as_index=False)
    .agg(
        n_traces=("position_label", "size"),
        median_improvement_factor_in_sample=("improvement_factor_in_sample", "median"),
        fraction_ramp_favored=("shape_call_in_sample", lambda values: float(np.mean(pd.Series(values) == "ramp-favored"))),
        fraction_ambiguous=("shape_call_in_sample", lambda values: float(np.mean(pd.Series(values) == "ambiguous"))),
        fraction_linear_favored=("shape_call_in_sample", lambda values: float(np.mean(pd.Series(values) == "linear-favored"))),
    )
)

observed_primary_summary_df = pd.DataFrame(
    [
        {
            "metric_label": "BMP4 positive fraction",
            "n_traces": int(len(primary_metric_df)),
            "median_improvement_factor_in_sample": float(primary_metric_df["improvement_factor_in_sample"].median()),
            "fraction_ramp_favored": float(np.mean(primary_metric_df["shape_call_in_sample"] == "ramp-favored")),
            "fraction_ambiguous": float(np.mean(primary_metric_df["shape_call_in_sample"] == "ambiguous")),
            "fraction_linear_favored": float(np.mean(primary_metric_df["shape_call_in_sample"] == "linear-favored")),
            "empirical_p_median_ratio": float(np.mean(null_simulation_summary_df["median_improvement_factor_in_sample"] >= primary_metric_df["improvement_factor_in_sample"].median())),
            "empirical_p_fraction_ramp_favored": float(np.mean(null_simulation_summary_df["fraction_ramp_favored"] >= np.mean(primary_metric_df["shape_call_in_sample"] == "ramp-favored"))),
        }
    ]
)

null_trace_summary_df = (
    null_trace_df.groupby("position_label", as_index=False)
    .agg(
        null_median_ratio=("improvement_factor_in_sample", "median"),
        null_ratio_q95=("improvement_factor_in_sample", lambda values: float(np.nanquantile(values, 0.95))),
    )
)
observed_vs_null_df = (
    primary_metric_df.merge(null_trace_summary_df, on="position_label", how="left")
    .sort_values("improvement_factor_in_sample", ascending=False)
    .reset_index(drop=True)
)
observed_vs_null_df["exceeds_trace_null_q95"] = observed_vs_null_df["improvement_factor_in_sample"] > observed_vs_null_df["null_ratio_q95"]

null_trace_path = TABLE_DIR / "05d_bmp4_positive_fraction_linear_null_by_trace.tsv"
null_trace_df.to_csv(null_trace_path, sep="\t", index=False)
null_summary_path = TABLE_DIR / "05d_bmp4_positive_fraction_linear_null_by_simulation.tsv"
null_simulation_summary_df.to_csv(null_summary_path, sep="\t", index=False)
observed_vs_null_path = TABLE_DIR / "05d_bmp4_positive_fraction_observed_vs_linear_null.tsv"
observed_vs_null_df.to_csv(observed_vs_null_path, sep="\t", index=False)

display(Markdown("**Observed positive-fraction summary versus the matched linear null**"))
display(observed_primary_summary_df.round(3))
display(Markdown("**Trace-level observed-versus-null comparison (top traces by observed ratio)**"))
display(
    observed_vs_null_df.loc[
        :,
        [
            "position_label",
            "scale_factor",
            "improvement_factor_in_sample",
            "shape_call_in_sample",
            "null_median_ratio",
            "null_ratio_q95",
            "exceeds_trace_null_q95",
        ],
    ]
    .head(12)
    .round(3)
)

print("Wrote table:", null_trace_path)
print("Wrote table:", null_summary_path)
print("Wrote table:", observed_vs_null_path)


##### Debug F. Individual Positive-Fraction Traces Versus A Matched Linear Null

This figure asks whether the observed line-versus-ramp advantage for `BMP4 positive fraction` is stronger than what we would expect if the underlying traces were truly linear apart from trace-specific residual structure.

The top row summarizes the null at the simulation level. The bottom-left panel compares each trace's observed in-sample ratio with its own null median. The bottom-right panel compares the observed ratio distribution with the pooled null distribution.


In [ ]:
linear_null_figure_path = FIGURE_DIR / "05d_bmp4_positive_fraction_linear_null.png"

fig, axes = plt.subplots(2, 2, figsize=(11.8, 8.0), constrained_layout=True)
observed_median_ratio = float(primary_metric_df["improvement_factor_in_sample"].median())
observed_fraction_ramp = float(np.mean(primary_metric_df["shape_call_in_sample"] == "ramp-favored"))

ax = axes[0, 0]
ax.hist(
    null_simulation_summary_df["median_improvement_factor_in_sample"].to_numpy(dtype=float),
    bins=18,
    color="0.75",
    edgecolor="white",
)
ax.axvline(observed_median_ratio, color="0.10", linewidth=2.2)
ax.axvline(1.0, color="0.55", linestyle="--", linewidth=1.0)
ax.set_xlabel("Simulation median linear / ramp RMSE")
ax.set_ylabel("Null simulation count")
ax.set_title("Simulation-level median ratio")
ax.grid(alpha=0.18)
ax.text(
    0.03,
    0.97,
    f"observed = {observed_median_ratio:.2f}\nempirical p = {observed_primary_summary_df['empirical_p_median_ratio'].iloc[0]:.3f}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=8.0,
    bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "0.8", "alpha": 0.92},
)

ax = axes[0, 1]
ax.hist(
    null_simulation_summary_df["fraction_ramp_favored"].to_numpy(dtype=float),
    bins=18,
    color="0.75",
    edgecolor="white",
)
ax.axvline(observed_fraction_ramp, color="0.10", linewidth=2.2)
ax.set_xlabel("Simulation fraction ramp-favored")
ax.set_ylabel("Null simulation count")
ax.set_title("Simulation-level ramp-favored fraction")
ax.grid(alpha=0.18)
ax.text(
    0.03,
    0.97,
    f"observed = {observed_fraction_ramp:.2f}\nempirical p = {observed_primary_summary_df['empirical_p_fraction_ramp_favored'].iloc[0]:.3f}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=8.0,
    bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "0.8", "alpha": 0.92},
)

ax = axes[1, 0]
scatter_df = observed_vs_null_df.loc[
    np.isfinite(observed_vs_null_df["improvement_factor_in_sample"].to_numpy(dtype=float))
    & np.isfinite(observed_vs_null_df["null_median_ratio"].to_numpy(dtype=float))
].copy()
if len(scatter_df):
    ax.scatter(
        scatter_df["null_median_ratio"],
        scatter_df["improvement_factor_in_sample"],
        c=np.log10(np.clip(scatter_df["scale_factor"].to_numpy(dtype=float), 1e-6, None)),
        cmap="viridis",
        s=28,
        alpha=0.9,
        edgecolor="none",
    )
    lower = float(np.nanmin([scatter_df["null_median_ratio"].min(), scatter_df["improvement_factor_in_sample"].min()]))
    upper = float(np.nanmax([scatter_df["null_median_ratio"].max(), scatter_df["improvement_factor_in_sample"].max()]))
    pad = 0.05 * (upper - lower if upper > lower else 1.0)
    ax.plot([lower - pad, upper + pad], [lower - pad, upper + pad], color="0.55", linestyle="--", linewidth=1.0)
    ax.set_xlim(lower - pad, upper + pad)
    ax.set_ylim(lower - pad, upper + pad)
ax.set_xlabel("Trace-specific null median ratio")
ax.set_ylabel("Observed trace ratio")
ax.set_title("Observed traces versus their own null median")
ax.grid(alpha=0.18)

ax = axes[1, 1]
observed_ratios = np.sort(primary_metric_df["improvement_factor_in_sample"].to_numpy(dtype=float))
observed_ratios = observed_ratios[np.isfinite(observed_ratios)]
null_ratios = np.sort(null_trace_df["improvement_factor_in_sample"].to_numpy(dtype=float))
null_ratios = null_ratios[np.isfinite(null_ratios)]
if observed_ratios.size:
    ax.step(
        observed_ratios,
        np.arange(1, observed_ratios.size + 1) / observed_ratios.size,
        where="post",
        color="0.10",
        linewidth=2.0,
        label="observed traces",
    )
if null_ratios.size:
    ax.step(
        null_ratios,
        np.arange(1, null_ratios.size + 1) / null_ratios.size,
        where="post",
        color="0.55",
        linewidth=1.8,
        label="pooled linear null",
    )
ax.axvline(1.0, color="0.55", linestyle="--", linewidth=1.0)
ax.set_xlabel("Linear / ramp RMSE")
ax.set_ylabel("Empirical CDF")
ax.set_title("Observed-versus-null ratio distribution")
ax.grid(alpha=0.18)
ax.legend(loc="lower right", frameon=False, fontsize=8.0)

fig.suptitle("Primary BMP4 positive-fraction traces: observed nonlinearity versus a matched linear null", fontsize=11.6)
fig.savefig(linear_null_figure_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", linear_null_figure_path)


#### Alignment To The YFP Transition

These plots unpack the same aligned-transition idea from Plot 1 in a more analysis-specific way.


##### Plot 1 (Repeated Here For Comparison). Absolute-Versus-Aligned YFP Sigmoid View


In [ ]:
display(Image(filename=str(sigmoid_figure_path)))


##### Plot 7. Aligned Positive-Fraction Means


In [ ]:
render_aligned_mean(
    aligned_fraction_df,
    FIGURE_DIR / "05d_yfp_halfmax_aligned_positive_fraction.png",
    "Aligned to YFP half-max: positive fraction (RFP 4 sigma, YFP 3 sigma masks)",
)


##### Plot 8. Aligned Positive-Region Mean-Intensity Means


In [ ]:
render_aligned_mean(
    aligned_mean_df,
    FIGURE_DIR / "05d_yfp_halfmax_aligned_positive_mean.png",
    "Aligned to YFP half-max: positive-region mean intensity (RFP 4 sigma, YFP 3 sigma masks)",
)


#### Position-Level Alignment Heterogeneity

This section keeps the position-level aligned view separate from the mean summaries above.


##### Plot 2 (Repeated Here For Comparison). Positive-Fraction Aligned Heatmaps


In [ ]:
display(Image(filename=str(aligned_heatmap_path)))


#### Rate Signatures After Alignment

These plots ask whether the aligned progression is not just sharper, but also dynamically cleaner to interpret.


##### Plot 9. Aligned Progression And Aligned Derivative Views

The top row shows the aligned progression.

The bottom row shows `d/dt` after Gaussian smoothing of the aggregated aligned mean (`sigma = 4` frames).


In [ ]:
aligned_metric_specs = [
    (
        "positive_fraction_rfp4_yfp3",
        aligned_fraction_df,
        "Positive fraction (RFP 4 sigma, YFP 3 sigma masks)",
        "normalized fraction per hour",
    ),
    (
        "positive_mean_intensity_rfp4_yfp3_z",
        aligned_mean_df,
        "Positive-region mean intensity (RFP 4 sigma, YFP 3 sigma masks)",
        "normalized intensity per hour",
    ),
]

fig, axes = plt.subplots(2, len(aligned_metric_specs), figsize=(5.4 * len(aligned_metric_specs), 7.0), sharex="col", constrained_layout=True)
for col_idx, (_, aligned_df_metric, metric_title, derivative_ylabel) in enumerate(aligned_metric_specs):
    top_ax = axes[0, col_idx]
    bottom_ax = axes[1, col_idx]
    top_plotted = []
    bottom_plotted = []
    for reporter in ["RFP", "YFP"]:
        summary = aggregate_aligned_mean(aligned_df_metric, reporter)
        x = summary["relative_hours"].to_numpy(dtype=float)
        y = summary["normalized_value"].to_numpy(dtype=float)
        y_smoothed = gaussian_smooth(y, 4.0)
        dydt = local_gradient(y_smoothed, x)
        top_plotted.append(y)
        bottom_plotted.append(dydt)
        top_ax.plot(x, y, color=REPORTER_COLORS[reporter], linewidth=2.5, label=reporter)
        bottom_ax.plot(x, dydt, color=REPORTER_COLORS[reporter], linewidth=2.5, label=reporter)

    top_ax.axvline(0.0, color="0.55", linestyle="--", linewidth=1.2)
    bottom_ax.axvline(0.0, color="0.55", linestyle="--", linewidth=1.2)
    bottom_ax.axhline(0.0, color="0.65", linestyle=":", linewidth=1.0)

    top_ax.set_title(metric_title, fontsize=10.0)
    top_ax.set_ylabel("Normalized progression")
    top_ax.set_ylim(focus_ylim_from_arrays(top_plotted, include_zero=True, padding_fraction=0.08))
    top_ax.grid(alpha=0.18)
    top_ax.legend(loc="upper left", frameon=False)

    bottom_ax.set_xlabel("Hours relative to YFP half-max")
    bottom_ax.set_ylabel(derivative_ylabel)
    bottom_ax.set_ylim(focus_ylim_from_arrays(bottom_plotted, include_zero=True, padding_fraction=0.12))
    bottom_ax.grid(alpha=0.18)

aligned_derivative_path = FIGURE_DIR / "05d_yfp_halfmax_aligned_derivative.png"
fig.suptitle("Aligned progression and aligned derivative views", fontsize=11.6)
fig.savefig(aligned_derivative_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", aligned_derivative_path)


##### Plot 10. Aligned YFP Rate-Versus-State View

This phase-plane style view focuses on `YFP` and compares several Gaussian smoothing choices for the derived rate estimate.


In [ ]:
derivative_parameter_choices = [
    {"smooth_param": 4.0, "color": "#4c78a8"},
    {"smooth_param": 8.0, "color": "#55a868"},
    {"smooth_param": 12.0, "color": "#c44e52"},
]

phase_plane_specs = [
    (
        aligned_fraction_df,
        "Positive fraction (RFP 4 sigma, YFP 3 sigma masks)",
        "normalized fraction per hour",
    ),
    (
        aligned_mean_df,
        "Positive-region mean intensity (RFP 4 sigma, YFP 3 sigma masks)",
        "normalized intensity per hour",
    ),
]

fig, axes = plt.subplots(1, len(phase_plane_specs), figsize=(5.4 * len(phase_plane_specs), 4.5), constrained_layout=True)
if len(phase_plane_specs) == 1:
    axes = np.asarray([axes])
for ax, (aligned_df_metric, metric_title, ylabel) in zip(axes, phase_plane_specs):
    plotted_rates = []
    for choice in derivative_parameter_choices:
        yfp_summary = aggregate_aligned_mean(aligned_df_metric, "YFP")
        x = yfp_summary["relative_hours"].to_numpy(dtype=float)
        y = yfp_summary["normalized_value"].to_numpy(dtype=float)
        y_smoothed = gaussian_smooth(y, choice["smooth_param"])
        rate = local_gradient(y_smoothed, x)
        finite = np.isfinite(y_smoothed) & np.isfinite(rate)
        if finite.sum() < 3:
            continue
        plotted_rates.append(rate[finite])
        ax.plot(
            y_smoothed[finite],
            rate[finite],
            color=choice["color"],
            linewidth=2.3,
            label=f"gaussian sigma={choice['smooth_param']:g}f",
        )
    ax.axhline(0.0, color="0.6", linestyle=":", linewidth=1.0)
    ax.set_title(metric_title, fontsize=10.0)
    ax.set_xlabel("Aligned YFP normalized progression state")
    ax.set_ylabel(ylabel)
    ax.set_ylim(focus_ylim_from_arrays(plotted_rates, include_zero=True, padding_fraction=0.12))
    ax.grid(alpha=0.18)
axes[-1].legend(loc="upper right", frameon=False, fontsize=8)
phase_plane_path = FIGURE_DIR / "05d_yfp_rate_vs_yfp_state.png"
fig.suptitle("Aligned YFP rate-versus-state view", fontsize=11.6)
fig.savefig(phase_plane_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", phase_plane_path)


#### Threshold Robustness Of The Aligned Readout

These plots keep the aligned-switching story honest by checking whether the main pattern depends too strongly on the exact threshold pairing.


##### Plot 11. Mixed-Threshold Positive-Fraction Alignment

This view uses the now-standard mixed pairing: `RFP = 4 sigma`, `YFP = 3 sigma`.


In [ ]:
mixed_relative_hours = np.arange(-24.0, 24.0 + 0.25, 0.25, dtype=float)
mixed_rows = []
mixed_lags = []
mixed_rfp_states = []
for position_label, _ in population_metrics.groupby("position_label", sort=True):
    yfp_anchor = halfmax_time_from_lookup(halfmax_lookup, "positive_fraction_sigma3", position_label, "YFP")
    rfp_halfmax = halfmax_time_from_lookup(halfmax_lookup, "positive_fraction_sigma4", position_label, "RFP")
    if np.isfinite(yfp_anchor) and np.isfinite(rfp_halfmax):
        mixed_lags.append(float(yfp_anchor - rfp_halfmax))
    if not np.isfinite(yfp_anchor):
        continue
    for reporter, metric_name in [("RFP", "positive_fraction_sigma4"), ("YFP", "positive_fraction_sigma3")]:
        subset = population_metrics.loc[
            (population_metrics["position_label"] == position_label)
            & (population_metrics["reporter"] == reporter)
        ].sort_values("time_hours")
        time_hours = subset["time_hours"].to_numpy(dtype=float)
        normalized_values = normalize_trace(subset[metric_name].to_numpy(dtype=float), early_n=8, smooth_sigma=4.0)
        finite = np.isfinite(time_hours) & np.isfinite(normalized_values)
        if finite.sum() < 12:
            continue
        valid_time = time_hours[finite]
        valid_values = normalized_values[finite]
        if reporter == "RFP" and yfp_anchor >= valid_time.min() and yfp_anchor <= valid_time.max():
            mixed_rfp_states.append(float(np.interp(yfp_anchor, valid_time, valid_values)))
        rel_time = valid_time - yfp_anchor
        if mixed_relative_hours.min() < rel_time.min() or mixed_relative_hours.max() > rel_time.max():
            continue
        interpolated = np.interp(mixed_relative_hours, rel_time, valid_values)
        for rel_h, interp_val in zip(mixed_relative_hours, interpolated):
            mixed_rows.append(
                {
                    "position_label": position_label,
                    "reporter": reporter,
                    "relative_hours": float(rel_h),
                    "normalized_value": float(interp_val),
                }
            )

mixed_df = pd.DataFrame(mixed_rows)
mixed_summary_df = pd.DataFrame(
    [
        {
            "rfp_metric_name": "positive_fraction_sigma4",
            "yfp_metric_name": "positive_fraction_sigma3",
            "n_lag_positions": int(len(mixed_lags)),
            "median_lag_hours": float(np.nanmedian(mixed_lags)) if mixed_lags else float("nan"),
            "fraction_yfp_after_rfp": float(np.mean(np.asarray(mixed_lags, dtype=float) > 0.0)) if mixed_lags else float("nan"),
            "n_state_positions": int(len(mixed_rfp_states)),
            "median_rfp_state_at_yfp_halfmax": float(np.nanmedian(mixed_rfp_states)) if mixed_rfp_states else float("nan"),
            "fraction_rfp_above_halfmax_when_yfp_hits_halfmax": float(np.mean(np.asarray(mixed_rfp_states, dtype=float) > 0.5)) if mixed_rfp_states else float("nan"),
        }
    ]
)
mixed_summary_path = TABLE_DIR / "05d_mixed_threshold_positive_fraction_summary.tsv"
mixed_summary_df.to_csv(mixed_summary_path, sep="\t", index=False)
mixed_fig_path = FIGURE_DIR / "05d_mixed_threshold_positive_fraction_alignment.png"
if mixed_df.empty:
    display(Markdown("No valid mixed-threshold aligned positive-fraction trajectories were available for plotting."))
else:
    fig, axes = plt.subplots(2, 1, figsize=(7.2, 6.8), sharex=True, constrained_layout=True)
    for reporter in ["RFP", "YFP"]:
        summary = (
            mixed_df.loc[mixed_df["reporter"] == reporter]
            .groupby("relative_hours", as_index=False)["normalized_value"]
            .mean()
            .sort_values("relative_hours")
        )
        x = summary["relative_hours"].to_numpy(dtype=float)
        y = summary["normalized_value"].to_numpy(dtype=float)
        y_smoothed = gaussian_smooth(y, 4.0)
        dydt = local_gradient(y_smoothed, x)
        axes[0].plot(x, y, color=REPORTER_COLORS[reporter], linewidth=2.5, label=reporter)
        axes[1].plot(x, dydt, color=REPORTER_COLORS[reporter], linewidth=2.5, label=reporter)

    axes[0].axvline(0.0, color="0.55", linestyle="--", linewidth=1.2)
    axes[1].axvline(0.0, color="0.55", linestyle="--", linewidth=1.2)
    axes[1].axhline(0.0, color="0.65", linestyle=":", linewidth=1.0)
    axes[0].set_ylabel("Normalized progression")
    axes[1].set_ylabel("Normalized fraction per hour")
    axes[1].set_xlabel("Hours relative to YFP (3 sigma) half-max")
    axes[0].legend(loc="upper left", frameon=False)
    axes[1].legend(loc="upper right", frameon=False)
    for ax in axes:
        ax.grid(alpha=0.18)

    fig.suptitle("Mixed threshold sensitivity: RFP 4 sigma, YFP 3 sigma", fontsize=11.6)
    fig.savefig(mixed_fig_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote figure:", mixed_fig_path)
display(display_time_df(mixed_summary_df))
print("Wrote table:", mixed_summary_path)


##### Plot 12. Threshold Sensitivity Of The Aligned YFP Rate-State View


In [ ]:
threshold_rate_specs = [
    ("positive_fraction_sigma{}", "Positive fraction", "normalized fraction per hour"),
    ("positive_mean_intensity_sigma{}_z", "Positive-region mean intensity", "normalized intensity per hour"),
]
threshold_colors = {2: "#9ecae1", 3: "#4c78a8", 4: "#fdae6b", 5: "#e6550d"}

fig, axes = plt.subplots(1, len(threshold_rate_specs), figsize=(5.4 * len(threshold_rate_specs), 4.4), constrained_layout=True)
if len(threshold_rate_specs) == 1:
    axes = np.asarray([axes])
for ax, (family_pattern, family_title, ylabel) in zip(axes, threshold_rate_specs):
    plotted = []
    for sigma_threshold in [2, 3, 4, 5]:
        metric_name = family_pattern.format(sigma_threshold)
        relative_hours = np.arange(-24.0, 24.0 + 0.25, 0.25, dtype=float)
        aligned_rows = []
        for position_label, _ in population_metrics.groupby("position_label", sort=True):
            anchor_time = halfmax_time_from_lookup(halfmax_lookup, metric_name, position_label, "YFP")
            if not np.isfinite(anchor_time):
                continue
            subset = population_metrics.loc[
                (population_metrics["position_label"] == position_label)
                & (population_metrics["reporter"] == "YFP")
            ].sort_values("time_hours")
            time_hours = subset["time_hours"].to_numpy(dtype=float)
            normalized_values = normalize_trace(subset[metric_name].to_numpy(dtype=float), early_n=8, smooth_sigma=4.0)
            finite = np.isfinite(time_hours) & np.isfinite(normalized_values)
            if finite.sum() < 12:
                continue
            rel_time = time_hours[finite] - anchor_time
            rel_values = normalized_values[finite]
            if relative_hours.min() < rel_time.min() or relative_hours.max() > rel_time.max():
                continue
            aligned_rows.append(np.interp(relative_hours, rel_time, rel_values))
        if not aligned_rows:
            continue
        aligned_matrix = np.vstack(aligned_rows)
        aligned_mean = np.nanmean(aligned_matrix, axis=0)
        state = gaussian_smooth(aligned_mean, 8.0)
        rate = local_gradient(state, relative_hours)
        finite = np.isfinite(state) & np.isfinite(rate)
        if finite.sum() < 3:
            continue
        plotted.append(rate[finite])
        ax.plot(
            state[finite],
            rate[finite],
            color=threshold_colors[sigma_threshold],
            linewidth=2.2,
            label=f"{sigma_threshold} sigma",
        )
    ax.axhline(0.0, color="0.55", linestyle=":", linewidth=1.0)
    ax.set_title(family_title, fontsize=10.0)
    ax.set_xlabel("Aligned YFP normalized progression state")
    ax.set_ylabel(ylabel)
    ax.set_ylim(focus_ylim_from_arrays(plotted, include_zero=True, padding_fraction=0.12))
    ax.grid(alpha=0.18)
axes[-1].legend(frameon=False)
threshold_rate_fig_path = FIGURE_DIR / "05d_threshold_sensitivity_rate_state.png"
fig.suptitle("Threshold sensitivity of the aligned YFP rate-state view", fontsize=11.6)
fig.savefig(threshold_rate_fig_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", threshold_rate_fig_path)
